# 08 - Recomendador híbrido final

Este notebook implementa un primer prototipo seguro del recomendador híbrido final. Sustituye LightFM como ranking principal y combina varias señales trazables:

- perfil de contenido explicable;
- filtrado colaborativo item-item basado en MovieLens;
- ratings y vistas reales de Trakt;
- filtros de calidad;
- diversidad;
- explicaciones breves de cada recomendación.

LightFM queda como experimento independiente en el notebook 07 y no se usa en este ranking final.

## 1. Imports y configuración

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("default")

RANDOM_STATE = 42

ROOT = Path("..")
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
REPORTS_RESULTADOS = ROOT / "reports" / "resultados"
TAGS_SEMANTIC_CLEAN_PATH = DATA_PROCESSED / "tags_semantic_clean.csv"
TAG_SEMANTIC_STATS_PATH = DATA_PROCESSED / "tag_semantic_stats.csv"

REPORTS_RESULTADOS.mkdir(parents=True, exist_ok=True)

## 2. Funciones auxiliares de carga

In [2]:
def require_file(path, message):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo esperado: {path}. {message}")
    return path


def load_csv_checked(path, message):
    return pd.read_csv(require_file(path, message))

## 3. Carga de datos

In [3]:
ratings = load_csv_checked(
    DATA_RAW / "ratings.csv",
    "Ejecuta 01_carga_datos.ipynb o verifica la carpeta data/raw.",
)
movies = load_csv_checked(
    DATA_PROCESSED / "movies_clean.csv",
    "Ejecuta 02_limpieza_transformacion.ipynb.",
)
trakt_ratings = load_csv_checked(
    DATA_PROCESSED / "trakt_ratings_mapped.csv",
    "Ejecuta 06_trakt_api_integracion.ipynb.",
)
trakt_watched = load_csv_checked(
    DATA_PROCESSED / "trakt_watched_mapped.csv",
    "Ejecuta 06_trakt_api_integracion.ipynb.",
)

datasets = {
    "ratings": ratings,
    "movies": movies,
    "trakt_ratings": trakt_ratings,
    "trakt_watched": trakt_watched,
}

for name, df in datasets.items():
    print(f"{name}: shape={df.shape}")
    print(f"Columnas: {list(df.columns)}\n")

print(f"Usuarios MovieLens: {ratings['userId'].nunique() if 'userId' in ratings.columns else 'columna userId no encontrada'}")
print(f"Películas MovieLens en ratings: {ratings['movieId'].nunique() if 'movieId' in ratings.columns else 'columna movieId no encontrada'}")
print(f"Ratings MovieLens: {len(ratings):,}")
print(f"Ratings Trakt mapeados: {len(trakt_ratings):,}")
print(f"Vistas Trakt mapeadas: {len(trakt_watched):,}")

ratings: shape=(32000204, 4)
Columnas: ['userId', 'movieId', 'rating', 'timestamp']

movies: shape=(87585, 26)
Columnas: ['movieId', 'title', 'genres', 'year', 'title_clean', 'rating_mean', 'rating_count', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']

trakt_ratings: shape=(147, 23)
Columnas: ['title', 'year', 'trakt_id', 'slug', 'imdb_id', 'tmdb_id', 'user_rating_trakt', 'user_rating_normalized', 'rated_at', 'plays', 'last_watched_at', 'imdbId_norm', 'tmdbId_norm', 'movieId_tmdb', 'movieId_imdb', 'movieId', 'match_source', 'title_ml', 'title_clean_ml', 'year_ml', 'genres_ml', 'rating_mean_ml', 'rating_count_ml']

trakt_watched: shape=(256, 23)
Columnas: ['title', 'year', 'trakt_id', 'slug', 'imdb_id', 'tmdb_id', 'user_rating_trakt', 'user_rating_normalized', 'rated_at', 'plays', 'last_watched_at', 'imdbId_norm', 'tmdbId_norm', 

## 4. Normalización defensiva de columnas

In [4]:
def _normalized_name(name):
    return str(name).strip().lower().replace("_", "").replace("-", "").replace(" ", "")


def find_column(df, candidates):
    exact = {str(col): col for col in df.columns}
    for candidate in candidates:
        if candidate in exact:
            return exact[candidate]
    normalized = {_normalized_name(col): col for col in df.columns}
    for candidate in candidates:
        col = normalized.get(_normalized_name(candidate))
        if col is not None:
            return col
    return None


def rename_first_match(df, target, candidates, required=True):
    if target in df.columns:
        return df
    source = find_column(df, candidates)
    if source is None:
        if required:
            raise KeyError(f"No se pudo encontrar una columna equivalente a '{target}'. Candidatas: {candidates}")
        return df
    return df.rename(columns={source: target})


ratings = rename_first_match(ratings, "userId", ["userId", "user_id", "userid"])
ratings = rename_first_match(ratings, "movieId", ["movieId", "movie_id", "movieid"])
ratings = rename_first_match(ratings, "rating", ["rating", "score"])

movies = rename_first_match(movies, "movieId", ["movieId", "movie_id", "movieid"])
movies = rename_first_match(movies, "title", ["title", "movie_title", "name"])
movies = rename_first_match(movies, "genres", ["genres", "genre", "movie_genres"])
movies = rename_first_match(movies, "year", ["year", "release_year"], required=False)
movies = rename_first_match(movies, "rating_mean", ["rating_mean", "mean_rating", "avg_rating", "average_rating"], required=False)
movies = rename_first_match(movies, "rating_count", ["rating_count", "num_ratings", "n_ratings", "rating_n"], required=False)

trakt_ratings = rename_first_match(trakt_ratings, "movieId", ["movieId", "movie_id", "movieid"])
trakt_watched = rename_first_match(trakt_watched, "movieId", ["movieId", "movie_id", "movieid"])

for df in [ratings, movies, trakt_ratings, trakt_watched]:
    df["movieId"] = pd.to_numeric(df["movieId"], errors="coerce")

ratings["userId"] = pd.to_numeric(ratings["userId"], errors="coerce")
ratings["rating"] = pd.to_numeric(ratings["rating"], errors="coerce")
ratings = ratings.dropna(subset=["userId", "movieId", "rating"]).copy()
ratings["userId"] = ratings["userId"].astype(int)
ratings["movieId"] = ratings["movieId"].astype(int)

movies = movies.dropna(subset=["movieId", "title", "genres"]).copy()
movies["movieId"] = movies["movieId"].astype(int)

if "year" not in movies.columns:
    movies["year"] = movies["title"].astype(str).str.extract(r"\((\d{4})\)", expand=False)
movies["year"] = pd.to_numeric(movies["year"], errors="coerce")

rating_stats = ratings.groupby("movieId")["rating"].agg(rating_mean_calc="mean", rating_count_calc="count").reset_index()
movies = movies.merge(rating_stats, on="movieId", how="left")
if "rating_mean" not in movies.columns:
    movies["rating_mean"] = movies["rating_mean_calc"]
else:
    movies["rating_mean"] = pd.to_numeric(movies["rating_mean"], errors="coerce").fillna(movies["rating_mean_calc"])
if "rating_count" not in movies.columns:
    movies["rating_count"] = movies["rating_count_calc"]
else:
    movies["rating_count"] = pd.to_numeric(movies["rating_count"], errors="coerce").fillna(movies["rating_count_calc"])
movies = movies.drop(columns=["rating_mean_calc", "rating_count_calc"])
movies["rating_count"] = movies["rating_count"].fillna(0).astype(int)

personal_rating_col = find_column(
    trakt_ratings,
    ["user_rating_normalized", "rating_normalized", "rating", "user_rating"],
)
if personal_rating_col is None:
    raise KeyError("No se encontró una columna de rating personal en Trakt.")

trakt_ratings["user_rating_5"] = pd.to_numeric(trakt_ratings[personal_rating_col], errors="coerce")
if trakt_ratings["user_rating_5"].dropna().max() > 5:
    trakt_ratings["user_rating_5"] = trakt_ratings["user_rating_5"] / 2.0
trakt_ratings["user_rating_5"] = trakt_ratings["user_rating_5"].clip(0, 5)

trakt_ratings = trakt_ratings.dropna(subset=["movieId", "user_rating_5"]).copy()
trakt_watched = trakt_watched.dropna(subset=["movieId"]).copy()
trakt_ratings["movieId"] = trakt_ratings["movieId"].astype(int)
trakt_watched["movieId"] = trakt_watched["movieId"].astype(int)

print(f"Columna de rating personal detectada: {personal_rating_col}")
print(f"Películas limpias disponibles: {len(movies):,}")

Columna de rating personal detectada: user_rating_normalized
Películas limpias disponibles: 80,505


## 5. Diagnóstico de perfil Trakt

In [5]:
# Diagnóstico de perfil Trakt

movie_profile_cols = ["movieId", "title", "genres", "year", "rating_mean", "rating_count"]
movie_profile_cols = [col for col in movie_profile_cols if col in movies.columns]

trakt_ratings_profile = trakt_ratings.merge(
    movies[movie_profile_cols],
    on="movieId",
    how="left",
    suffixes=("_trakt", "_movie")
)

# Resolver columnas duplicadas tras el merge
if "title" not in trakt_ratings_profile.columns:
    if "title_movie" in trakt_ratings_profile.columns:
        trakt_ratings_profile["title"] = trakt_ratings_profile["title_movie"]
    elif "title_trakt" in trakt_ratings_profile.columns:
        trakt_ratings_profile["title"] = trakt_ratings_profile["title_trakt"]
    else:
        trakt_ratings_profile["title"] = "movieId=" + trakt_ratings_profile["movieId"].astype(str)

if "genres" not in trakt_ratings_profile.columns:
    if "genres_movie" in trakt_ratings_profile.columns:
        trakt_ratings_profile["genres"] = trakt_ratings_profile["genres_movie"]
    elif "genres_trakt" in trakt_ratings_profile.columns:
        trakt_ratings_profile["genres"] = trakt_ratings_profile["genres_trakt"]
    else:
        trakt_ratings_profile["genres"] = ""

# Asegurar user_rating_5
if "user_rating_5" not in trakt_ratings_profile.columns:
    raise KeyError(
        "No existe la columna user_rating_5. Revisa la celda anterior de normalización de ratings de Trakt."
    )

liked_movies = trakt_ratings_profile[trakt_ratings_profile["user_rating_5"] >= 4.0].copy()
neutral_movies = trakt_ratings_profile[
    (trakt_ratings_profile["user_rating_5"] > 2.5)
    & (trakt_ratings_profile["user_rating_5"] < 4.0)
].copy()
disliked_movies = trakt_ratings_profile[trakt_ratings_profile["user_rating_5"] <= 2.5].copy()

print(f"Películas gustadas: {len(liked_movies)}")
print(f"Películas neutras: {len(neutral_movies)}")
print(f"Películas no gustadas: {len(disliked_movies)}")

if len(liked_movies) < 3:
    warnings.warn("Perfil positivo muy pequeño; las recomendaciones pueden ser poco estables.")

display_cols = ["title", "user_rating_5"]
display(liked_movies.sort_values("user_rating_5", ascending=False)[display_cols].head(10))
display(disliked_movies.sort_values("user_rating_5")[display_cols].head(10))

Películas gustadas: 47
Películas neutras: 56
Películas no gustadas: 20


,title,user_rating_5
96,Interstellar (2014),5.0
26,Toy Story 3 (2010),4.5
38,Toy Story 2 (1999),4.5
33,Shrek 2 (2004),4.5
112,No Country for Old Men (2007),4.5
82,Donnie Darko (2001),4.5
23,Once Upon a Time in Hollywood (2019),4.5
104,Monty Python's Life of Brian (1979),4.5
88,Shutter Island (2010),4.5
110,Arrival (2016),4.5


,title,user_rating_5
74,Now You See Me (2013),1.0
91,Aftersun (2022),1.0
86,Get Out (2017),1.5
10,My Fault (2023),1.5
73,"Matrix Revolutions, The (2003)",2.0
75,The Revenant (2015),2.0
64,Bullet Train (2022),2.0
66,Gravity (2013),2.0
76,Iron Man (2008),2.0
63,"Maze Runner, The (2014)",2.0


## 6. Preparación de matriz colaborativa item-item

In [6]:
valid_movie_ids = set(movies["movieId"].dropna().astype(int))
popular_movie_ids = set(movies.loc[movies["rating_count"] >= 50, "movieId"].astype(int))

ratings_cf = ratings[ratings["movieId"].isin(valid_movie_ids & popular_movie_ids)].copy()
user_counts = ratings_cf.groupby("userId").size()
eligible_users = user_counts[user_counts >= 20].index
ratings_cf = ratings_cf[ratings_cf["userId"].isin(eligible_users)].copy()

if ratings_cf.empty:
    warnings.warn("La matriz colaborativa queda vacía tras los filtros de calidad. Revisa ratings.csv y movies_clean.csv.")
    user_item_matrix = sparse.csr_matrix((0, 0), dtype=np.float32)
    movie_id_to_col = {}
    col_to_movie_id = {}
    movie_id_to_title = {}
else:
    ratings_cf["mean_rating_user"] = ratings_cf.groupby("userId")["rating"].transform("mean")
    ratings_cf["rating_centered"] = ratings_cf["rating"] - ratings_cf["mean_rating_user"]

    user_codes, user_ids = pd.factorize(ratings_cf["userId"], sort=True)
    movie_codes, movie_ids = pd.factorize(ratings_cf["movieId"], sort=True)

    user_item_matrix = sparse.csr_matrix(
        (ratings_cf["rating_centered"].astype(np.float32), (user_codes, movie_codes)),
        shape=(len(user_ids), len(movie_ids)),
        dtype=np.float32,
    )

    movie_id_to_col = {int(movie_id): int(col) for col, movie_id in enumerate(movie_ids)}
    col_to_movie_id = {int(col): int(movie_id) for col, movie_id in enumerate(movie_ids)}
    movie_id_to_title = movies.set_index("movieId")["title"].to_dict()

matrix_cells = user_item_matrix.shape[0] * user_item_matrix.shape[1]
density = user_item_matrix.nnz / matrix_cells if matrix_cells else 0
print(f"Dimensiones matriz usuario-película: {user_item_matrix.shape}")
print(f"Interacciones no nulas: {user_item_matrix.nnz:,}")
print(f"Densidad aproximada: {density:.6f}")

Dimensiones matriz usuario-película: (200526, 15947)
Interacciones no nulas: 31,460,128
Densidad aproximada: 0.009838


## 7. Función de similitud item-item bajo demanda

In [7]:
SIMILAR_MOVIES_COLUMNS = ["movieId", "similarity", "title", "genres", "year", "rating_mean", "rating_count"]


def get_similar_movies_item_item(
    target_movie_id,
    user_item_matrix,
    movie_id_to_col,
    col_to_movie_id,
    movies_df,
    top_n=50,
    min_similarity=0.05,
):
    target_movie_id = int(target_movie_id)
    if target_movie_id not in movie_id_to_col or user_item_matrix.shape[1] == 0:
        return pd.DataFrame(columns=SIMILAR_MOVIES_COLUMNS)

    target_col = movie_id_to_col[target_movie_id]
    target_vector = user_item_matrix[:, target_col].T
    similarities = cosine_similarity(target_vector, user_item_matrix.T).ravel()

    result = pd.DataFrame(
        {
            "movieId": [col_to_movie_id[col] for col in range(len(similarities))],
            "similarity": similarities,
        }
    )
    result = result[(result["movieId"] != target_movie_id) & (result["similarity"] > min_similarity)]
    result = result.sort_values("similarity", ascending=False).head(top_n)

    metadata_cols = ["movieId", "title", "genres", "year", "rating_mean", "rating_count"]
    result = result.merge(movies_df[metadata_cols], on="movieId", how="left")
    return result[SIMILAR_MOVIES_COLUMNS]

## 8. Sanity check de similitudes

In [8]:
liked_available = liked_movies[liked_movies["movieId"].isin(movie_id_to_col.keys())].head(3)

if liked_available.empty:
    warnings.warn("Ninguna película gustada está disponible en la matriz colaborativa item-item.")
else:
    for _, liked_row in liked_available.iterrows():
        title = liked_row.get("title", f"movieId={liked_row['movieId']}")
        print(f"\nPelícula base: {title}")
        similar = get_similar_movies_item_item(
            liked_row["movieId"],
            user_item_matrix,
            movie_id_to_col,
            col_to_movie_id,
            movies,
            top_n=10,
            min_similarity=0.05,
        )
        display(similar)


Película base: Cinema Paradiso (Nuovo cinema Paradiso) (1989)


,movieId,similarity,title,genres,year,rating_mean,rating_count
0,912,0.155976,Casablanca (1942),Drama|Romance,1942.0,4.194517,32190
1,1225,0.155691,Amadeus (1984),Drama,1984.0,4.072641,24986
2,1131,0.151308,Jean de Florette (1986),Drama|Mystery,1986.0,4.103292,3524
3,1193,0.148326,One Flew Over the Cuckoo's Nest (1975),Drama,1975.0,4.204229,44592
4,1207,0.147705,To Kill a Mockingbird (1962),Drama,1962.0,4.134469,18785
5,904,0.143327,Rear Window (1954),Mystery|Thriller,1954.0,4.226701,24883
6,2324,0.139511,Life Is Beautiful (La Vita è bella) (1997),Comedy|Drama|Romance|War,1997.0,4.150374,29819
7,1132,0.139169,Manon of the Spring (Manon des sources) (1986),Drama,1986.0,4.063254,3233
8,58,0.136045,"Postman, The (Postino, Il) (1994)",Comedy|Drama|Romance,1994.0,3.967750,11659
9,1247,0.135198,"Graduate, The (1967)",Comedy|Drama|Romance,1967.0,4.021081,22319



Película base: City of God (Cidade de Deus) (2002)


,movieId,similarity,title,genres,year,rating_mean,rating_count
0,2959,0.224863,Fight Club (1999),Action|Crime|Drama|Thriller,1999.0,4.228780,77332
1,2329,0.222359,American History X (1998),Crime|Drama,1998.0,4.130444,38967
2,1213,0.219109,Goodfellas (1990),Crime|Drama,1990.0,4.188427,42003
3,4226,0.217764,Memento (2000),Mystery|Thriller,2000.0,4.141601,53658
4,44555,0.217555,"Lives of Others, The (Das leben der Anderen) (...",Drama|Romance|Thriller,2006.0,4.198770,12273
5,858,0.217434,"Godfather, The (1972)",Crime|Drama,1972.0,4.317030,66440
6,4235,0.215905,Amores Perros (Love's a Bitch) (2000),Drama|Thriller,2000.0,3.992665,7703
7,27773,0.213912,Old Boy (2003),Mystery|Thriller,2003.0,4.089464,17331
8,1221,0.209897,"Godfather: Part II, The (1974)",Crime|Drama,1974.0,4.264468,43111
9,7361,0.201881,Eternal Sunshine of the Spotless Mind (2004),Drama|Romance|Sci-Fi,2004.0,4.065294,44430



Película base: Edward Scissorhands (1990)


,movieId,similarity,title,genres,year,rating_mean,rating_count
0,551,0.204613,"Nightmare Before Christmas, The (1993)",Animation|Children|Fantasy|Musical,1993.0,3.742149,26081
1,2174,0.173241,Beetlejuice (1988),Comedy|Fantasy,1988.0,3.516450,24681
2,7361,0.132263,Eternal Sunshine of the Spotless Mind (2004),Drama|Romance|Sci-Fi,2004.0,4.065294,44430
3,7147,0.131229,Big Fish (2003),Drama|Fantasy|Romance,2003.0,3.798002,22072
4,4878,0.129285,Donnie Darko (2001),Drama|Mystery|Sci-Fi|Thriller,2001.0,3.935533,35452
5,4973,0.127355,"Amelie (Fabuleux destin d'Amélie Poulain, Le) ...",Comedy|Romance,2001.0,4.082169,43459
6,2997,0.126717,Being John Malkovich (1999),Comedy|Drama|Fantasy,1999.0,3.933888,36151
7,1258,0.126577,"Shining, The (1980)",Horror,1980.0,4.036439,38640
8,1206,0.124933,"Clockwork Orange, A (1971)",Crime|Drama|Sci-Fi|Thriller,1971.0,3.985442,36818
9,1682,0.123761,"Truman Show, The (1998)",Comedy|Drama|Sci-Fi,1998.0,3.891833,44140


## 9. Score colaborativo personalizado

In [9]:
COLLAB_COLUMNS = [
    "movieId",
    "collab_raw_score",
    "collab_positive_raw",
    "item_item_collab_score",
    "item_item_negative_collab_score",
    "positive_collab_score",
    "negative_collab_score",
    "n_collab_evidence",
    "similar_liked_movies",
    "similar_disliked_movies",
]


def _short_unique_titles(titles, max_titles=3):
    clean_titles = []
    for title in titles:
        if pd.notna(title) and str(title) not in clean_titles:
            clean_titles.append(str(title))
        if len(clean_titles) >= max_titles:
            break
    return ", ".join(clean_titles)


def _robust_minmax(series):
    series = pd.to_numeric(series, errors="coerce").fillna(0)
    if series.empty:
        return series
    low, high = np.nanpercentile(series, [5, 95])
    if np.isclose(low, high):
        return pd.Series(np.where(series > 0, 1.0, 0.0), index=series.index)
    return ((series - low) / (high - low)).clip(0, 1)


def compute_item_item_collab_scores(
    user_ratings_df,
    user_item_matrix,
    movie_id_to_col,
    col_to_movie_id,
    movies_df,
    top_n_per_seed=200,
    min_similarity=0.03,
):
    if user_ratings_df.empty or user_item_matrix.shape[1] == 0:
        return pd.DataFrame(columns=COLLAB_COLUMNS)

    seed_titles = movies_df.set_index("movieId")["title"].to_dict()
    aggregates = {}

    for _, user_row in user_ratings_df.dropna(subset=["movieId", "user_rating_5"]).iterrows():
        seed_movie_id = int(user_row["movieId"])
        if seed_movie_id not in movie_id_to_col:
            continue

        personal_weight = float(user_row["user_rating_5"]) - 3.0
        if np.isclose(personal_weight, 0):
            continue

        similar_movies = get_similar_movies_item_item(
            seed_movie_id,
            user_item_matrix,
            movie_id_to_col,
            col_to_movie_id,
            movies_df,
            top_n=top_n_per_seed,
            min_similarity=min_similarity,
        )
        seed_title = seed_titles.get(seed_movie_id, str(seed_movie_id))

        for _, similar_row in similar_movies.iterrows():
            candidate_id = int(similar_row["movieId"])
            contribution = float(similar_row["similarity"]) * personal_weight
            candidate = aggregates.setdefault(
                candidate_id,
                {
                    "collab_raw_score": 0.0,
                    "positive_collab_score": 0.0,
                    "negative_collab_score": 0.0,
                    "n_collab_evidence": 0,
                    "similar_liked_movies": [],
                    "similar_disliked_movies": [],
                },
            )
            candidate["collab_raw_score"] += contribution
            candidate["n_collab_evidence"] += 1
            if contribution > 0:
                candidate["positive_collab_score"] += contribution
                candidate["similar_liked_movies"].append(seed_title)
            elif contribution < 0:
                candidate["negative_collab_score"] += abs(contribution)
                candidate["similar_disliked_movies"].append(seed_title)

    if not aggregates:
        return pd.DataFrame(columns=COLLAB_COLUMNS)

    rows = []
    for movie_id, values in aggregates.items():
        rows.append(
            {
                "movieId": movie_id,
                "collab_raw_score": values["collab_raw_score"],
                "positive_collab_score": values["positive_collab_score"],
                "negative_collab_score": values["negative_collab_score"],
                "n_collab_evidence": values["n_collab_evidence"],
                "similar_liked_movies": _short_unique_titles(values["similar_liked_movies"]),
                "similar_disliked_movies": _short_unique_titles(values["similar_disliked_movies"]),
            }
        )

    scores = pd.DataFrame(rows)
    scores["collab_positive_raw"] = scores["collab_raw_score"].clip(lower=0)
    scores["item_item_collab_score"] = scores["collab_positive_raw"].rank(pct=True)
    scores.loc[scores["collab_positive_raw"] <= 0, "item_item_collab_score"] = 0
    scores["item_item_negative_collab_score"] = scores["negative_collab_score"].rank(pct=True)
    scores.loc[scores["negative_collab_score"] <= 0, "item_item_negative_collab_score"] = 0
    return scores[COLLAB_COLUMNS]


collab_scores = compute_item_item_collab_scores(
    trakt_ratings,
    user_item_matrix,
    movie_id_to_col,
    col_to_movie_id,
    movies,
)

print(f"Películas con señal colaborativa personalizada: {len(collab_scores):,}")
display(collab_scores.sort_values("item_item_collab_score", ascending=False).head(10))

Películas con señal colaborativa personalizada: 3,309


,movieId,collab_raw_score,collab_positive_raw,item_item_collab_score,item_item_negative_collab_score,positive_collab_score,negative_collab_score,n_collab_evidence,similar_liked_movies,similar_disliked_movies
200,2959,6.422709,6.422709,1.000000,0.904503,6.627034,0.204324,57,"City of God (Cidade de Deus) (2002), Dallas Bu...","Kick-Ass (2010), The Revenant (2015), Get Out ..."
93,4226,6.329096,6.329096,0.999698,0.921426,6.570744,0.241648,61,Cinema Paradiso (Nuovo cinema Paradiso) (1989)...,"Gravity (2013), Kick-Ass (2010), The Revenant ..."
58,1213,5.755333,5.755333,0.999396,0.894228,5.945597,0.190264,55,Cinema Paradiso (Nuovo cinema Paradiso) (1989)...,"The Revenant (2015), Dead Alive (Braindead) (1..."
105,296,5.611456,5.611456,0.999093,0.871260,5.774753,0.163297,48,Cinema Paradiso (Nuovo cinema Paradiso) (1989)...,"The Revenant (2015), Get Out (2017)"
10,858,5.593983,5.593983,0.998791,0.702025,5.655693,0.061710,53,Cinema Paradiso (Nuovo cinema Paradiso) (1989)...,The Revenant (2015)
125,1089,5.537346,5.537346,0.998489,0.896041,5.731123,0.193777,51,Cinema Paradiso (Nuovo cinema Paradiso) (1989)...,"The Revenant (2015), Dead Alive (Braindead) (1..."
203,7361,5.513310,5.513310,0.998187,0.906618,5.726481,0.213171,59,"City of God (Cidade de Deus) (2002), Dallas Bu...","Gravity (2013), The Revenant (2015), Get Out (..."
55,318,5.468758,5.468758,0.997885,0.907223,5.682389,0.213630,54,Cinema Paradiso (Nuovo cinema Paradiso) (1989)...,Harry Potter and the Deathly Hallows: Part 2 (...
24,1221,5.463455,5.463455,0.997582,0.689030,5.521255,0.057799,52,Cinema Paradiso (Nuovo cinema Paradiso) (1989)...,The Revenant (2015)
3,1193,5.354545,5.354545,0.997280,0.734361,5.432955,0.078410,49,Cinema Paradiso (Nuovo cinema Paradiso) (1989)...,"The Revenant (2015), Dead Alive (Braindead) (1..."


## 10. Candidatos base y filtros

In [10]:
rated_movie_ids = set(trakt_ratings["movieId"].dropna().astype(int))
watched_movie_ids = set(trakt_watched["movieId"].dropna().astype(int))
excluded_movie_ids = rated_movie_ids | watched_movie_ids

base_candidates = movies.copy()
base_candidates["genres_text"] = base_candidates["genres"].fillna("").astype(str).str.strip()

candidate_mask = (
    ~base_candidates["movieId"].isin(excluded_movie_ids)
    & base_candidates["year"].notna()
    & base_candidates["genres_text"].ne("")
    & base_candidates["genres_text"].ne("(no genres listed)")
    & (base_candidates["rating_mean"] >= 3.2)
    & (base_candidates["rating_count"] >= 300)
)
candidates = base_candidates[candidate_mask].copy()

if len(candidates) < 100:
    warnings.warn("Quedan menos de 100 candidatos con rating_count >= 300; se relaja rating_count a 100.")
    candidate_mask = (
        ~base_candidates["movieId"].isin(excluded_movie_ids)
        & base_candidates["year"].notna()
        & base_candidates["genres_text"].ne("")
        & base_candidates["genres_text"].ne("(no genres listed)")
        & (base_candidates["rating_mean"] >= 3.2)
        & (base_candidates["rating_count"] >= 100)
    )
    candidates = base_candidates[candidate_mask].copy()

print(f"Candidatos tras filtros: {len(candidates):,}")

Candidatos tras filtros: 4,863


## 11. Score de contenido simple y explicable

In [11]:
def split_genres(genres):
    if pd.isna(genres):
        return []
    genres = str(genres).strip()
    if not genres or genres == "(no genres listed)":
        return []
    return [genre.strip() for genre in genres.split("|") if genre.strip()]


def build_genre_weights(profile_df):
    counts = {}
    for genres in profile_df["genres"].dropna():
        for genre in split_genres(genres):
            counts[genre] = counts.get(genre, 0) + 1
    if not counts:
        return {}
    max_count = max(counts.values())
    return {genre: count / max_count for genre, count in counts.items()}


def genre_affinity(genres, weights):
    genre_list = split_genres(genres)
    if not genre_list or not weights:
        return 0.0
    score = sum(weights.get(genre, 0.0) for genre in genre_list)
    normalizer = sum(weights.values()) if weights else 1.0
    return float(np.clip(score / normalizer, 0, 1))


positive_genre_weights = build_genre_weights(liked_movies)
negative_genre_weights = build_genre_weights(disliked_movies)

tag_columns = [col for col in ["tags", "tags_clean", "tags_features", "tags_features_en", "tag_features"] if col in movies.columns]
if tag_columns:
    print(f"Columnas de tags detectadas para fase futura, no usadas en este prototipo: {tag_columns}")

candidates["genre_profile_score"] = candidates["genres"].apply(lambda value: genre_affinity(value, positive_genre_weights))
candidates["negative_genre_score"] = candidates["genres"].apply(lambda value: genre_affinity(value, negative_genre_weights))
candidates["content_profile_score"] = candidates["genre_profile_score"]
candidates["negative_similarity_score"] = candidates["negative_genre_score"]

print("Pesos positivos de géneros:")
print(positive_genre_weights)
print("Pesos negativos de géneros:")
print(negative_genre_weights)

Pesos positivos de géneros:
{'Drama': 1.0, 'Action': 0.25, 'Adventure': 0.2857142857142857, 'Crime': 0.35714285714285715, 'Thriller': 0.42857142857142855, 'Fantasy': 0.2857142857142857, 'Romance': 0.25, 'Comedy': 0.75, 'Animation': 0.21428571428571427, 'Children': 0.21428571428571427, 'IMAX': 0.07142857142857142, 'Horror': 0.07142857142857142, 'Musical': 0.03571428571428571, 'Sci-Fi': 0.39285714285714285, 'War': 0.07142857142857142, 'Mystery': 0.07142857142857142}
Pesos negativos de géneros:
{'Drama': 1.0, 'Romance': 0.375, 'Action': 0.875, 'Adventure': 0.625, 'Fantasy': 0.5, 'Mystery': 0.5, 'IMAX': 0.5, 'Comedy': 0.625, 'Thriller': 0.625, 'Sci-Fi': 0.5, 'Crime': 0.25, 'Horror': 0.25}


## Perfil semántico del usuario con tags/genome de MovieLens

In [12]:
import re


tags_path = DATA_RAW / "tags.csv"
genome_scores_path = DATA_RAW / "genome-scores.csv"
genome_tags_path = DATA_RAW / "genome-tags.csv"

semantic_files = {
    "tags_semantic_clean.csv": TAGS_SEMANTIC_CLEAN_PATH.exists(),
    "tag_semantic_stats.csv": TAG_SEMANTIC_STATS_PATH.exists(),
    "tags.csv": tags_path.exists(),
    "genome-scores.csv": genome_scores_path.exists(),
    "genome-tags.csv": genome_tags_path.exists(),
}
print("Archivos semánticos disponibles:")
for filename, exists in semantic_files.items():
    print(f"- {filename}: {'sí' if exists else 'no'}")

if TAGS_SEMANTIC_CLEAN_PATH.exists() and TAG_SEMANTIC_STATS_PATH.exists():
    semantic_source = "processed_tags"
elif tags_path.exists():
    warnings.warn("No existen tags semánticos procesados. Ejecuta primero notebooks/09_preprocesado_tags_semanticos.ipynb.")
    semantic_source = "raw_tags_fallback"
else:
    semantic_source = "none"
print(f"Fuente semántica usada: {semantic_source}")

SEMANTIC_COLUMNS = [
    "semantic_raw_score",
    "negative_semantic_raw_score",
    "semantic_profile_score",
    "negative_semantic_score",
    "semantic_explanation_terms",
    "negative_semantic_terms",
]

candidates["semantic_raw_score"] = 0.0
candidates["negative_semantic_raw_score"] = 0.0
candidates["semantic_profile_score"] = 0.0
candidates["negative_semantic_score"] = 0.0
candidates["semantic_explanation_terms"] = ""
candidates["negative_semantic_terms"] = ""
top_positive_semantic_terms = []
top_negative_semantic_terms = []
positive_tag_profile = pd.DataFrame(columns=["tag_clean", "positive_weight"])
negative_tag_profile = pd.DataFrame(columns=["tag_clean", "negative_weight"])


def normalize_positive_rank(raw_scores):
    normalized = pd.Series(0.0, index=raw_scores.index, dtype=float)
    positive_mask = raw_scores.fillna(0) > 0
    if positive_mask.any():
        normalized.loc[positive_mask] = raw_scores.loc[positive_mask].rank(pct=True)
    return normalized


def top_terms_from_contrib(contrib_df, term_col="tag", value_col="term_contribution", max_terms=5):
    if contrib_df.empty:
        return pd.Series(dtype=str)
    ordered = contrib_df.sort_values(["movieId", value_col], ascending=[True, False])
    return ordered.groupby("movieId")[term_col].apply(
        lambda values: ", ".join(values.astype(str).drop_duplicates().head(max_terms))
    )


def normalize_tag_text(tag):
    if pd.isna(tag):
        return None
    tag_clean = str(tag).lower().strip()
    tag_clean = tag_clean.strip('"\'`´“”‘’')
    tag_clean = tag_clean.replace("_", " ")
    protected_hyphen_terms = {"thought-provoking", "post-apocalyptic"}
    if tag_clean not in protected_hyphen_terms:
        tag_clean = re.sub(r"\s*-\s*", " ", tag_clean)
    tag_clean = re.sub(r"\s+", " ", tag_clean).strip()
    tag_clean = tag_clean.strip('"\'`´“”‘’')
    if not tag_clean or tag_clean in {"nan", "none", "null"}:
        return None
    return tag_clean


def is_bad_tag(tag_clean):
    if tag_clean is None or not str(tag_clean).strip():
        return True
    tag_clean = str(tag_clean).strip()
    if len(tag_clean) < 3:
        return True
    if tag_clean.isnumeric():
        return True
    if re.fullmatch(r"[\W_]+", tag_clean):
        return True
    if re.search(r"\b(18|19|20)\d{2}\b", tag_clean):
        return True
    if re.fullmatch(r"\d{1,2}[/-]\d{1,2}([/-]\d{2,4})?", tag_clean):
        return True
    if "http" in tag_clean or "www." in tag_clean:
        return True
    if not re.search(r"[a-z]", tag_clean):
        return True

    exact_whitelist = {
        "great soundtrack", "visuals", "visually appealing", "dark comedy", "black comedy",
        "social commentary", "twist ending", "thought-provoking", "mindfuck", "time travel",
        "artificial intelligence", "post-apocalyptic", "coming of age", "atmospheric", "surreal",
        "psychological", "dystopia", "dreamlike", "stylized", "cinematography", "soundtrack",
        "bittersweet", "quirky", "nonlinear",
    }
    administrative_patterns = [
        "imdb", "oscar", "academy award", "criterion", "netflix", "dvd", "blu", "owned",
        "watchlist", "want to see", "seen", "watched", "top 250", "1001 movies", "afi",
        "award nominated", "award winner", "bd ", "bd-",
    ]
    if any(pattern in tag_clean for pattern in administrative_patterns):
        return True
    if tag_clean in exact_whitelist:
        return False

    exact_blacklist = {
        "owned", "own", "seen", "watched", "watchlist", "want to see", "to watch", "dvd",
        "bd-r", "bd r", "bdr", "blu-ray", "blu ray", "bluray", "blue-ray", "blue ray",
        "netflix", "amazon", "hulu", "criterion", "criterion collection", "library", "collection",
        "on dvr", "recorded", "vhs", "tv", "tivo", "imdb", "imdb top 250", "top 250",
        "top 100", "afi", "afi 100", "1001 movies", "1001 movies you must see before you die",
        "must see", "classic", "classics", "cult classic", "oscar", "oscars", "oscar winner",
        "oscar nominated", "academy award", "academy awards", "award winner", "award nominated",
        "best picture", "golden globe", "palme d'or", "good", "great", "best", "bad", "worst",
        "boring", "funny", "very funny", "hilarious", "overrated", "underrated", "favorite",
        "favourite", "favorites", "favourites", "excellent", "amazing", "awesome", "terrible",
        "awful", "mediocre", "masterpiece", "movie", "movies", "film", "films", "cinema",
        "cinematic", "based on", "based on book", "based on a book", "adapted from", "adaptation",
        "remake", "sequel", "franchise", "original", "drama", "comedy", "action", "thriller",
        "romance", "horror", "sci-fi", "science fiction", "crime", "adventure", "animation",
        "children", "fantasy", "mystery", "documentary", "war", "western", "musical", "film-noir",
        "film noir", "noir", "f word", "f-word",
    }
    if tag_clean in exact_blacklist:
        return True
    return False


def build_clean_tags_dataframe(tags_raw):
    tags_clean = tags_raw.copy()
    original_rows = len(tags_clean)
    original_unique_tags = tags_clean["tag"].nunique(dropna=True) if "tag" in tags_clean.columns else 0
    tags_clean["tag_clean"] = tags_clean["tag"].apply(normalize_tag_text)
    tags_clean["bad_tag"] = tags_clean["tag_clean"].apply(is_bad_tag)
    tags_clean = tags_clean[tags_clean["tag_clean"].notna() & ~tags_clean["bad_tag"]].copy()
    keep_cols = [col for col in ["userId", "movieId", "tag", "tag_clean", "timestamp"] if col in tags_clean.columns]
    tags_clean = tags_clean[keep_cols].copy()
    tags_clean["movieId"] = pd.to_numeric(tags_clean["movieId"], errors="coerce")
    tags_clean = tags_clean.dropna(subset=["movieId"])
    tags_clean["movieId"] = tags_clean["movieId"].astype(int)
    if "userId" in tags_clean.columns:
        tags_clean["userId"] = pd.to_numeric(tags_clean["userId"], errors="coerce")
        tags_clean = tags_clean.drop_duplicates(["userId", "movieId", "tag_clean"])
    else:
        tags_clean = tags_clean.drop_duplicates(["movieId", "tag_clean"])
    removed_rows = original_rows - len(tags_clean)
    removed_pct = removed_rows / original_rows * 100 if original_rows else 0
    print(f"Filas originales tags.csv: {original_rows:,}")
    print(f"Tags únicos originales: {original_unique_tags:,}")
    print(f"Filas tras limpieza básica: {len(tags_clean):,}")
    print(f"Tags únicos tras limpieza: {tags_clean['tag_clean'].nunique():,}")
    print(f"Filas eliminadas: {removed_rows:,} ({removed_pct:.2f}%)")
    return tags_clean


def build_tag_statistics(tags_clean):
    n_movies_total = tags_clean["movieId"].nunique()
    stats = tags_clean.groupby("tag_clean").agg(n_uses=("movieId", "size"), n_movies=("movieId", "nunique")).reset_index()
    if "userId" in tags_clean.columns:
        stats_users = tags_clean.groupby("tag_clean")["userId"].nunique().rename("n_users").reset_index()
        user_tag_counts = tags_clean.groupby(["tag_clean", "userId"]).size().rename("user_uses").reset_index()
        top_user = user_tag_counts.groupby("tag_clean")["user_uses"].max().rename("top_user_uses").reset_index()
        stats = stats.merge(stats_users, on="tag_clean", how="left").merge(top_user, on="tag_clean", how="left")
        stats["top_user_share"] = stats["top_user_uses"] / stats["n_uses"]
    else:
        stats["n_users"] = np.nan
        stats["top_user_uses"] = np.nan
        stats["top_user_share"] = 0.0
    stats["pct_movies"] = stats["n_movies"] / n_movies_total if n_movies_total else 0
    stats["idf"] = np.log((1 + n_movies_total) / (1 + stats["n_movies"])) + 1
    low_movies = stats["n_movies"] < 5
    low_users = stats["n_users"] < 5 if "userId" in tags_clean.columns else pd.Series(False, index=stats.index)
    high_pct_movies = stats["pct_movies"] > 0.05
    high_top_user_share = stats["top_user_share"] > 0.60 if "userId" in tags_clean.columns else pd.Series(False, index=stats.index)
    stats["is_reliable_tag"] = ~(low_movies | low_users | high_pct_movies | high_top_user_share)
    print(f"Tags antes del filtro de fiabilidad: {len(stats):,}")
    print(f"Tags fiables: {stats['is_reliable_tag'].sum():,}")
    print(f"Eliminados por n_movies bajo: {low_movies.sum():,}")
    print(f"Eliminados por n_users bajo: {low_users.sum():,}")
    print(f"Eliminados por pct_movies alto: {high_pct_movies.sum():,}")
    print(f"Eliminados por top_user_share alto: {high_top_user_share.sum():,}")
    reliable = stats[stats["is_reliable_tag"]].copy()
    display(reliable.sort_values("n_uses", ascending=False).head(30))
    display(reliable.sort_values("n_movies", ascending=False).head(30))
    display(reliable[reliable["n_movies"] >= 5].sort_values("idf", ascending=False).head(30))
    display(stats[high_top_user_share].sort_values("top_user_share", ascending=False).head(30))
    return stats


def build_semantic_profiles_from_tags(tags_clean, liked_movies, disliked_movies, tag_stats):
    idf_map = tag_stats.set_index("tag_clean")["idf"]
    positive_seed = liked_movies.loc[liked_movies["user_rating_5"] >= 4.0, ["movieId", "user_rating_5"]].dropna().copy()
    negative_seed = disliked_movies.loc[disliked_movies["user_rating_5"] <= 2.5, ["movieId", "user_rating_5"]].dropna().copy()
    positive_seed["movieId"] = positive_seed["movieId"].astype(int)
    negative_seed["movieId"] = negative_seed["movieId"].astype(int)

    positive_profile = pd.DataFrame(columns=["tag_clean", "positive_weight", "positive_n_seed_movies"])
    if not positive_seed.empty:
        positive_tags = tags_clean.merge(positive_seed, on="movieId", how="inner")
        positive_tags["idf"] = positive_tags["tag_clean"].map(idf_map).fillna(1.0)
        positive_tags["personal_weight"] = positive_tags["user_rating_5"] - 3.0
        positive_tags["tag_weight"] = positive_tags["personal_weight"] * positive_tags["idf"]
        positive_profile = positive_tags.groupby("tag_clean").agg(
            positive_weight=("tag_weight", "sum"),
            positive_n_seed_movies=("movieId", "nunique"),
        ).reset_index()
        positive_profile = positive_profile[positive_profile["positive_n_seed_movies"] >= 1]
        positive_profile = positive_profile.sort_values("positive_weight", ascending=False).head(80)

    negative_profile = pd.DataFrame(columns=["tag_clean", "negative_weight", "negative_n_seed_movies"])
    if not negative_seed.empty:
        negative_tags = tags_clean.merge(negative_seed, on="movieId", how="inner")
        negative_tags["idf"] = negative_tags["tag_clean"].map(idf_map).fillna(1.0)
        negative_tags["personal_weight"] = 3.0 - negative_tags["user_rating_5"]
        negative_tags["tag_weight"] = negative_tags["personal_weight"] * negative_tags["idf"]
        negative_profile = negative_tags.groupby("tag_clean").agg(
            negative_weight=("tag_weight", "sum"),
            negative_n_seed_movies=("movieId", "nunique"),
        ).reset_index()
        negative_profile = negative_profile.sort_values("negative_weight", ascending=False).head(80)

    overlap = sorted(set(positive_profile["tag_clean"]) & set(negative_profile["tag_clean"]))
    if overlap:
        positive_profile.loc[positive_profile["tag_clean"].isin(overlap), "positive_weight"] *= 0.5
        negative_profile.loc[negative_profile["tag_clean"].isin(overlap), "negative_weight"] *= 0.5
        positive_profile = positive_profile.sort_values("positive_weight", ascending=False)
        negative_profile = negative_profile.sort_values("negative_weight", ascending=False)

    display(positive_profile.head(30))
    display(negative_profile.head(30))
    print(f"Tags en intersección positivo/negativo: {overlap[:30]}")
    print(f"Número de tags positivos: {len(positive_profile)}")
    print(f"Número de tags negativos: {len(negative_profile)}")
    return positive_profile, negative_profile


def build_semantic_profiles_from_processed_tags(tags_clean, liked_movies, disliked_movies):
    required_columns = {"movieId", "tag_clean", "idf"}
    if not required_columns.issubset(tags_clean.columns):
        missing = sorted(required_columns - set(tags_clean.columns))
        raise ValueError(f"tags_semantic_clean.csv no tiene las columnas esperadas: {missing}")

    profile_tags = tags_clean[["movieId", "tag_clean", "idf"]].copy()
    profile_tags["movieId"] = pd.to_numeric(profile_tags["movieId"], errors="coerce")
    profile_tags["idf"] = pd.to_numeric(profile_tags["idf"], errors="coerce").fillna(1.0)
    profile_tags = profile_tags.dropna(subset=["movieId", "tag_clean"])
    profile_tags["movieId"] = profile_tags["movieId"].astype(int)
    profile_tags = profile_tags.drop_duplicates(["movieId", "tag_clean"])

    positive_seed = liked_movies.loc[liked_movies["user_rating_5"] >= 4.0, ["movieId", "user_rating_5"]].dropna().copy()
    negative_seed = disliked_movies.loc[disliked_movies["user_rating_5"] <= 2.5, ["movieId", "user_rating_5"]].dropna().copy()
    positive_seed["movieId"] = positive_seed["movieId"].astype(int)
    negative_seed["movieId"] = negative_seed["movieId"].astype(int)

    positive_profile = pd.DataFrame(columns=["tag_clean", "positive_weight", "positive_n_seed_movies"])
    if not positive_seed.empty:
        positive_tags = profile_tags.merge(positive_seed, on="movieId", how="inner")
        positive_tags["tag_weight"] = (positive_tags["user_rating_5"] - 3.0) * positive_tags["idf"]
        positive_profile = positive_tags.groupby("tag_clean", as_index=False).agg(
            positive_weight=("tag_weight", "sum"),
            positive_n_seed_movies=("movieId", "nunique"),
        )

    negative_profile = pd.DataFrame(columns=["tag_clean", "negative_weight", "negative_n_seed_movies"])
    if not negative_seed.empty:
        negative_tags = profile_tags.merge(negative_seed, on="movieId", how="inner")
        negative_tags["tag_weight"] = (3.0 - negative_tags["user_rating_5"]) * negative_tags["idf"]
        negative_profile = negative_tags.groupby("tag_clean", as_index=False).agg(
            negative_weight=("tag_weight", "sum"),
            negative_n_seed_movies=("movieId", "nunique"),
        )

    overlap = sorted(set(positive_profile["tag_clean"]) & set(negative_profile["tag_clean"]))
    if overlap:
        ambiguity_penalty = 0.25
        positive_profile.loc[positive_profile["tag_clean"].isin(overlap), "positive_weight"] *= ambiguity_penalty
        negative_profile.loc[negative_profile["tag_clean"].isin(overlap), "negative_weight"] *= ambiguity_penalty

    positive_profile = positive_profile.sort_values("positive_weight", ascending=False).head(80)
    negative_profile = negative_profile.sort_values("negative_weight", ascending=False).head(80)

    display(positive_profile.head(30))
    display(negative_profile.head(30))
    print(f"Tags en intersección positivo/negativo: {overlap[:30]}")
    print(f"Número de tags positivos: {len(positive_profile)}")
    print(f"Número de tags negativos: {len(negative_profile)}")
    return positive_profile, negative_profile


def compute_semantic_scores_from_tags(candidates, tags_clean, positive_tag_profile, negative_tag_profile):
    result = candidates[["movieId"]].copy()
    result["semantic_raw_score"] = 0.0
    result["negative_semantic_raw_score"] = 0.0
    result["semantic_profile_score"] = 0.0
    result["negative_semantic_score"] = 0.0
    result["semantic_explanation_terms"] = ""
    result["negative_semantic_terms"] = ""
    candidate_tags = tags_clean[tags_clean["movieId"].isin(result["movieId"].astype(int))][["movieId", "tag_clean"]].drop_duplicates()
    if candidate_tags.empty:
        return result

    if not positive_tag_profile.empty:
        positive_contrib = candidate_tags.merge(positive_tag_profile[["tag_clean", "positive_weight"]], on="tag_clean", how="inner")
        positive_contrib["contribution"] = positive_contrib["positive_weight"]
        raw_positive = positive_contrib.groupby("movieId")["contribution"].sum()
        result["semantic_raw_score"] = result["movieId"].map(raw_positive).fillna(0.0)
        result["semantic_profile_score"] = normalize_positive_rank(result["semantic_raw_score"])
        result["semantic_explanation_terms"] = result["movieId"].map(
            top_terms_from_contrib(positive_contrib, term_col="tag_clean", value_col="contribution", max_terms=5)
        ).fillna("")

    if not negative_tag_profile.empty:
        negative_contrib = candidate_tags.merge(negative_tag_profile[["tag_clean", "negative_weight"]], on="tag_clean", how="inner")
        negative_contrib["contribution"] = negative_contrib["negative_weight"]
        raw_negative = negative_contrib.groupby("movieId")["contribution"].sum()
        result["negative_semantic_raw_score"] = result["movieId"].map(raw_negative).fillna(0.0)
        result["negative_semantic_score"] = normalize_positive_rank(result["negative_semantic_raw_score"])
        result["negative_semantic_terms"] = result["movieId"].map(
            top_terms_from_contrib(negative_contrib, term_col="tag_clean", value_col="contribution", max_terms=5)
        ).fillna("")
    return result


def compute_semantic_scores_from_processed_tags(candidates, tags_clean, positive_tag_profile, negative_tag_profile):
    return compute_semantic_scores_from_tags(candidates, tags_clean, positive_tag_profile, negative_tag_profile)


positive_seed_ratings = liked_movies.loc[liked_movies["user_rating_5"] >= 4.0, ["movieId", "user_rating_5"]].dropna().copy()
negative_seed_ratings = disliked_movies.loc[disliked_movies["user_rating_5"] <= 2.5, ["movieId", "user_rating_5"]].dropna().copy()
positive_seed_ratings["movieId"] = positive_seed_ratings["movieId"].astype(int)
negative_seed_ratings["movieId"] = negative_seed_ratings["movieId"].astype(int)

if semantic_source == "processed_tags":
    tags_clean = pd.read_csv(TAGS_SEMANTIC_CLEAN_PATH)
    tag_stats = pd.read_csv(TAG_SEMANTIC_STATS_PATH)
    required_clean_cols = {"movieId", "tag_clean", "idf"}
    required_stats_cols = {"tag_clean", "idf", "is_reliable_tag"}
    if not required_clean_cols.issubset(tags_clean.columns) or not required_stats_cols.issubset(tag_stats.columns):
        warnings.warn("Los tags semánticos procesados no tienen las columnas esperadas; se dejan scores semánticos en 0.")
        semantic_source = "none"
    else:
        tags_clean = tags_clean.copy()
        tags_clean["movieId"] = pd.to_numeric(tags_clean["movieId"], errors="coerce")
        tags_clean["idf"] = pd.to_numeric(tags_clean["idf"], errors="coerce").fillna(1.0)
        tags_clean = tags_clean.dropna(subset=["movieId", "tag_clean"])
        tags_clean["movieId"] = tags_clean["movieId"].astype(int)
        tags_clean = tags_clean.drop_duplicates(["movieId", "tag_clean"])
        print(f"Tags limpios cargados: {len(tags_clean):,}")
        print(f"Tags únicos limpios: {tags_clean['tag_clean'].nunique():,}")
        print(f"Películas con tags limpios: {tags_clean['movieId'].nunique():,}")
        positive_tag_profile, negative_tag_profile = build_semantic_profiles_from_processed_tags(tags_clean, liked_movies, disliked_movies)
        semantic_scores = compute_semantic_scores_from_processed_tags(candidates, tags_clean, positive_tag_profile, negative_tag_profile)
        candidates = candidates.drop(columns=[col for col in SEMANTIC_COLUMNS if col in candidates.columns]).merge(
            semantic_scores, on="movieId", how="left"
        )
        for col in ["semantic_raw_score", "negative_semantic_raw_score", "semantic_profile_score", "negative_semantic_score"]:
            candidates[col] = candidates[col].fillna(0.0)
        for col in ["semantic_explanation_terms", "negative_semantic_terms"]:
            candidates[col] = candidates[col].fillna("")
        top_positive_semantic_terms = positive_tag_profile["tag_clean"].dropna().head(20).tolist()
        top_negative_semantic_terms = negative_tag_profile["tag_clean"].dropna().head(20).tolist()

elif semantic_source == "genome":
    genome_scores = pd.read_csv(genome_scores_path)
    genome_tags = pd.read_csv(genome_tags_path)
    required_genome_cols = {"movieId", "tagId", "relevance"}
    required_tag_cols = {"tagId", "tag"}
    if not required_genome_cols.issubset(genome_scores.columns) or not required_tag_cols.issubset(genome_tags.columns):
        warnings.warn("Los archivos genome no tienen las columnas esperadas; se dejan scores semánticos en 0.")
        semantic_source = "none"
    else:
        genome_scores["movieId"] = pd.to_numeric(genome_scores["movieId"], errors="coerce")
        genome_scores["tagId"] = pd.to_numeric(genome_scores["tagId"], errors="coerce")
        genome_scores["relevance"] = pd.to_numeric(genome_scores["relevance"], errors="coerce")
        genome_scores = genome_scores.dropna(subset=["movieId", "tagId", "relevance"]).copy()
        genome_scores["movieId"] = genome_scores["movieId"].astype(int)
        genome_scores["tagId"] = genome_scores["tagId"].astype(int)
        genome_tags["tagId"] = pd.to_numeric(genome_tags["tagId"], errors="coerce")
        genome_tags = genome_tags.dropna(subset=["tagId", "tag"]).copy()
        genome_tags["tagId"] = genome_tags["tagId"].astype(int)

        relevant_movie_ids = set(candidates["movieId"].astype(int)) | set(positive_seed_ratings["movieId"]) | set(negative_seed_ratings["movieId"])
        genome_scores = genome_scores[genome_scores["movieId"].isin(relevant_movie_ids)].copy()
        candidate_genome = genome_scores[genome_scores["movieId"].isin(candidates["movieId"].astype(int))].copy()

        positive_profile = pd.DataFrame(columns=["tagId", "tag_weight"])
        if not positive_seed_ratings.empty:
            positive_profile = genome_scores.merge(positive_seed_ratings, on="movieId", how="inner")
            positive_profile["personal_weight"] = positive_profile["user_rating_5"] - 3.0
            positive_profile["weighted_relevance"] = positive_profile["relevance"] * positive_profile["personal_weight"]
            positive_profile = positive_profile.groupby("tagId", as_index=False)["weighted_relevance"].sum()
            positive_profile = positive_profile.sort_values("weighted_relevance", ascending=False).head(100)
            max_positive = positive_profile["weighted_relevance"].max()
            positive_profile["tag_weight"] = positive_profile["weighted_relevance"] / max_positive if max_positive > 0 else 0

        negative_profile = pd.DataFrame(columns=["tagId", "tag_weight"])
        if not negative_seed_ratings.empty:
            negative_profile = genome_scores.merge(negative_seed_ratings, on="movieId", how="inner")
            negative_profile["personal_weight"] = 3.0 - negative_profile["user_rating_5"]
            negative_profile["weighted_relevance"] = negative_profile["relevance"] * negative_profile["personal_weight"]
            negative_profile = negative_profile.groupby("tagId", as_index=False)["weighted_relevance"].sum()
            negative_profile = negative_profile.sort_values("weighted_relevance", ascending=False).head(100)
            max_negative = negative_profile["weighted_relevance"].max()
            negative_profile["tag_weight"] = negative_profile["weighted_relevance"] / max_negative if max_negative > 0 else 0

        if not positive_profile.empty and not candidate_genome.empty:
            positive_contrib = candidate_genome.merge(positive_profile[["tagId", "tag_weight"]], on="tagId", how="inner")
            positive_contrib["term_contribution"] = positive_contrib["relevance"] * positive_contrib["tag_weight"]
            raw_positive = positive_contrib.groupby("movieId")["term_contribution"].sum()
            candidates["semantic_raw_score"] = candidates["movieId"].map(raw_positive).fillna(0.0)
            candidates["semantic_profile_score"] = normalize_positive_rank(candidates["semantic_raw_score"])
            positive_terms = positive_contrib.merge(genome_tags, on="tagId", how="left")
            candidates["semantic_explanation_terms"] = candidates["movieId"].map(top_terms_from_contrib(positive_terms)).fillna("")
            top_positive_semantic_terms = positive_profile.merge(genome_tags, on="tagId", how="left")["tag"].dropna().head(15).tolist()

        if not negative_profile.empty and not candidate_genome.empty:
            negative_contrib = candidate_genome.merge(negative_profile[["tagId", "tag_weight"]], on="tagId", how="inner")
            negative_contrib["term_contribution"] = negative_contrib["relevance"] * negative_contrib["tag_weight"]
            raw_negative = negative_contrib.groupby("movieId")["term_contribution"].sum()
            candidates["negative_semantic_raw_score"] = candidates["movieId"].map(raw_negative).fillna(0.0)
            candidates["negative_semantic_score"] = normalize_positive_rank(candidates["negative_semantic_raw_score"])
            negative_terms = negative_contrib.merge(genome_tags, on="tagId", how="left")
            candidates["negative_semantic_terms"] = candidates["movieId"].map(top_terms_from_contrib(negative_terms)).fillna("")
            top_negative_semantic_terms = negative_profile.merge(genome_tags, on="tagId", how="left")["tag"].dropna().head(15).tolist()

elif semantic_source == "raw_tags_fallback":
    tags_raw = pd.read_csv(tags_path)
    if not {"movieId", "tag"}.issubset(tags_raw.columns):
        warnings.warn("tags.csv no tiene las columnas esperadas; se dejan scores semánticos en 0.")
        semantic_source = "none"
    else:
        tags_clean = build_clean_tags_dataframe(tags_raw)
        tag_stats = build_tag_statistics(tags_clean)
        reliable_tags = set(tag_stats.loc[tag_stats["is_reliable_tag"], "tag_clean"])
        tags_clean = tags_clean[tags_clean["tag_clean"].isin(reliable_tags)].copy()
        print(f"Filas finales de tags fiables: {len(tags_clean):,}")
        print(f"Tags únicos finales: {tags_clean['tag_clean'].nunique():,}")
        print(f"Películas con al menos un tag fiable: {tags_clean['movieId'].nunique():,}")
        positive_tag_profile, negative_tag_profile = build_semantic_profiles_from_tags(tags_clean, liked_movies, disliked_movies, tag_stats)
        semantic_scores = compute_semantic_scores_from_tags(candidates, tags_clean, positive_tag_profile, negative_tag_profile)
        candidates = candidates.drop(columns=[col for col in SEMANTIC_COLUMNS if col in candidates.columns]).merge(
            semantic_scores, on="movieId", how="left"
        )
        for col in ["semantic_raw_score", "negative_semantic_raw_score", "semantic_profile_score", "negative_semantic_score"]:
            candidates[col] = candidates[col].fillna(0.0)
        for col in ["semantic_explanation_terms", "negative_semantic_terms"]:
            candidates[col] = candidates[col].fillna("")
        top_positive_semantic_terms = positive_tag_profile["tag_clean"].dropna().head(20).tolist()
        top_negative_semantic_terms = negative_tag_profile["tag_clean"].dropna().head(20).tolist()

if semantic_source == "none":
    warnings.warn("No hay genome-scores/genome-tags ni tags.csv; los scores semánticos se dejan en 0.")

for col in ["semantic_raw_score", "negative_semantic_raw_score", "semantic_profile_score", "negative_semantic_score"]:
    candidates[col] = candidates[col].fillna(0.0)
for col in ["semantic_explanation_terms", "negative_semantic_terms"]:
    candidates[col] = candidates[col].fillna("")

FORBIDDEN_SEMANTIC_TERMS = [
    "hans zimmer", "tom hanks", "coen brothers", "christopher nolan", "pixar",
    "great acting", "good acting", "cult film", "inspirational", "touching", "slow paced",
]
found_forbidden_terms = set()
if "tag_clean" in positive_tag_profile.columns:
    found_forbidden_terms.update(set(positive_tag_profile["tag_clean"].dropna()) & set(FORBIDDEN_SEMANTIC_TERMS))
if "tag_clean" in negative_tag_profile.columns:
    found_forbidden_terms.update(set(negative_tag_profile["tag_clean"].dropna()) & set(FORBIDDEN_SEMANTIC_TERMS))
semantic_terms_text = " ".join(candidates["semantic_explanation_terms"].fillna("").astype(str).str.lower())
for term in FORBIDDEN_SEMANTIC_TERMS:
    if term in semantic_terms_text:
        found_forbidden_terms.add(term)
if found_forbidden_terms:
    print("Advertencia: siguen apareciendo tags que deberían haber sido filtrados por el 09.")
    print(sorted(found_forbidden_terms))

positive_semantic_count = (candidates["semantic_profile_score"] > 0).sum()
negative_semantic_count = (candidates["negative_semantic_score"] > 0).sum()
print(f"Fuente semántica usada final: {semantic_source}")
print(f"Candidatos con semantic_profile_score > 0: {positive_semantic_count:,} ({positive_semantic_count / len(candidates) * 100:.2f}%)")
print(f"Candidatos con negative_semantic_score > 0: {negative_semantic_count:,} ({negative_semantic_count / len(candidates) * 100:.2f}%)")
print("Top 20 tags positivos finales:")
print(top_positive_semantic_terms[:20])
print("Top 20 tags negativos finales:")
print(top_negative_semantic_terms[:20])


Archivos semánticos disponibles:
- tags_semantic_clean.csv: sí
- tag_semantic_stats.csv: sí
- tags.csv: sí
- genome-scores.csv: no
- genome-tags.csv: no
Fuente semántica usada: processed_tags
Tags limpios cargados: 50,897
Tags únicos limpios: 534
Películas con tags limpios: 16,795


C:\Users\alexo\AppData\Local\Temp\ipykernel_27880\1710737712.py:347: DtypeWarning: Columns (0: semantic_category, 1: semantic_match_reason, 2: rejection_reason_statistical) have mixed types. Specify dtype option on import or set low_memory=False.
  tag_stats = pd.read_csv(TAG_SEMANTIC_STATS_PATH)


,tag_clean,positive_weight,positive_n_seed_movies
232,tense,64.307744,12
182,psychology,54.468484,10
157,nostalgic,53.348209,7
93,existentialism,52.521941,8
12,ambiguous ending,52.335802,6
77,dreamlike,50.012098,8
229,surrealism,49.732575,10
238,time travel,46.161323,8
152,neo-noir,46.075157,8
237,time loop,45.113847,5


,tag_clean,negative_weight,negative_n_seed_movies
51,long takes,22.442782,2
83,shaky camera,14.091218,1
91,stylized violence,11.987656,2
39,gory,9.819927,2
98,tragic hero,9.819927,2
55,memory loss,9.401857,2
53,massachusetts institute of technology,9.031525,1
56,military industrial complex,8.032996,1
38,giant robots,8.032996,1
86,space program,7.932912,1


Tags en intersección positivo/negativo: ['absurd', 'absurdism', 'afterlife', 'amazing photography', 'android(s)/cyborg(s)', 'apocalypse', 'artificial intelligence', 'atmospheric', 'bad ending', 'black comedy', 'brutal', 'brutal violence', 'cinematography', 'class differences', 'colorful', 'coming of age', 'dark', 'dark comedy', 'dark hero', 'dark humor', 'disappointing ending', 'disturbing', 'dry humor', 'dystopia', 'emotional', 'ending', 'existential', 'existential crisis', 'faith', 'family relationships']
Número de tags positivos: 80
Número de tags negativos: 80
Fuente semántica usada final: processed_tags
Candidatos con semantic_profile_score > 0: 2,893 (59.49%)
Candidatos con negative_semantic_score > 0: 3,137 (64.51%)
Top 20 tags positivos finales:
['tense', 'psychology', 'nostalgic', 'existentialism', 'ambiguous ending', 'dreamlike', 'surrealism', 'time travel', 'neo-noir', 'time loop', 'creepy', 'morality', 'depressing', 'psychological', 'heartwarming', 'hope', 'confusing', 'bit

## 12. Scores de calidad y popularidad

In [13]:
candidates["rating_score"] = ((candidates["rating_mean"] - 3.0) / 2.0).clip(0, 1)

log_counts = np.log1p(candidates["rating_count"].clip(lower=0))
if np.isclose(log_counts.max(), log_counts.min()):
    candidates["popularity_score"] = 0.0
else:
    candidates["popularity_score"] = ((log_counts - log_counts.min()) / (log_counts.max() - log_counts.min())).clip(0, 1)

display(candidates[["title", "rating_mean", "rating_count", "rating_score", "popularity_score"]].head())

,title,rating_mean,rating_count,rating_score,popularity_score
0,Jumanji (1995),3.275758,28904,0.137879,0.782331
1,Heat (1995),3.868277,29490,0.434139,0.785770
2,Sabrina (1995),3.363968,13585,0.181984,0.652937
3,GoldenEye (1995),3.427850,32474,0.213925,0.802290
4,"American President, The (1995)",3.657160,19127,0.328580,0.711571


## 13. Fusionar señales

In [14]:
candidates_scored = candidates.merge(collab_scores, on="movieId", how="left")

numeric_collab_cols = [
    "collab_raw_score",
    "collab_positive_raw",
    "item_item_collab_score",
    "item_item_negative_collab_score",
    "positive_collab_score",
    "negative_collab_score",
    "n_collab_evidence",
]
for col in numeric_collab_cols:
    candidates_scored[col] = candidates_scored[col].fillna(0)

for col in ["similar_liked_movies", "similar_disliked_movies"]:
    candidates_scored[col] = candidates_scored[col].fillna("")

print(f"Candidatos con scoring fusionado: {len(candidates_scored):,}")

Candidatos con scoring fusionado: 4,863


## Diagnóstico adaptativo del perfil de usuario

In [15]:
def _clip01(value):
    return float(np.clip(value, 0.0, 1.0))


def _rank_positive(raw_scores):
    normalized = pd.Series(0.0, index=raw_scores.index, dtype=float)
    positive_mask = raw_scores.fillna(0.0) > 0
    if positive_mask.any():
        normalized.loc[positive_mask] = raw_scores.loc[positive_mask].rank(pct=True)
    return normalized


def _top_terms_from_weighted_tags(contrib_df, term_col="tag_clean", value_col="contribution", max_terms=5):
    if contrib_df.empty:
        return pd.Series(dtype=str)
    ordered = contrib_df.sort_values(["movieId", value_col], ascending=[True, False])
    return ordered.groupby("movieId")[term_col].apply(
        lambda values: ", ".join(values.astype(str).drop_duplicates().head(max_terms))
    )


def build_discriminative_semantic_profile(
    tags_clean,
    liked_movies,
    disliked_movies,
    positive_tag_profile=None,
    negative_tag_profile=None,
    smoothing=0.10,
    min_positive_seed_movies=1,
    beta_negative=0.75,
):
    required_cols = {"movieId", "tag_clean", "idf"}
    if tags_clean is None or tags_clean.empty or not required_cols.issubset(tags_clean.columns):
        return pd.DataFrame(columns=[
            "tag_clean", "semantic_category", "positive_weight", "negative_weight",
            "positive_weight_norm", "negative_weight_norm", "positive_n_seed_movies",
            "negative_n_seed_movies", "tag_ambiguity", "discriminative_lift",
            "discriminative_weight_raw", "discriminative_weight",
            "negative_discriminative_weight", "is_core_positive_tag", "is_core_negative_tag",
        ])

    tag_cols = ["movieId", "tag_clean", "idf"] + (["semantic_category"] if "semantic_category" in tags_clean.columns else [])
    tag_features = tags_clean[tag_cols].copy()
    tag_features["movieId"] = pd.to_numeric(tag_features["movieId"], errors="coerce")
    tag_features["idf"] = pd.to_numeric(tag_features["idf"], errors="coerce").fillna(1.0)
    tag_features = tag_features.dropna(subset=["movieId", "tag_clean"])
    tag_features["movieId"] = tag_features["movieId"].astype(int)
    tag_features = tag_features.drop_duplicates(["movieId", "tag_clean"])

    positive_seed = liked_movies.loc[liked_movies["user_rating_5"] >= 4.0, ["movieId", "user_rating_5"]].dropna().copy()
    negative_seed = disliked_movies.loc[disliked_movies["user_rating_5"] <= 2.5, ["movieId", "user_rating_5"]].dropna().copy()
    positive_seed["movieId"] = positive_seed["movieId"].astype(int)
    negative_seed["movieId"] = negative_seed["movieId"].astype(int)

    positive_agg = pd.DataFrame(columns=["tag_clean", "positive_weight", "positive_n_seed_movies", "semantic_category"])
    if not positive_seed.empty:
        positive_tags = tag_features.merge(positive_seed, on="movieId", how="inner")
        positive_tags["positive_personal_weight"] = positive_tags["user_rating_5"] - 3.0
        positive_tags["contribution_positive"] = positive_tags["positive_personal_weight"] * positive_tags["idf"]
        agg_spec = {
            "positive_weight": ("contribution_positive", "sum"),
            "positive_n_seed_movies": ("movieId", "nunique"),
        }
        if "semantic_category" in positive_tags.columns:
            agg_spec["semantic_category"] = ("semantic_category", "first")
        positive_agg = positive_tags.groupby("tag_clean", as_index=False).agg(**agg_spec)
        if "semantic_category" not in positive_agg.columns:
            positive_agg["semantic_category"] = None

    negative_agg = pd.DataFrame(columns=["tag_clean", "negative_weight", "negative_n_seed_movies"])
    if not negative_seed.empty:
        negative_tags = tag_features.merge(negative_seed, on="movieId", how="inner")
        negative_tags["negative_personal_weight"] = 3.0 - negative_tags["user_rating_5"]
        negative_tags["contribution_negative"] = negative_tags["negative_personal_weight"] * negative_tags["idf"]
        negative_agg = negative_tags.groupby("tag_clean", as_index=False).agg(
            negative_weight=("contribution_negative", "sum"),
            negative_n_seed_movies=("movieId", "nunique"),
        )

    profile = positive_agg.merge(negative_agg, on="tag_clean", how="outer")
    if profile.empty:
        return pd.DataFrame(columns=[
            "tag_clean", "semantic_category", "positive_weight", "negative_weight",
            "positive_weight_norm", "negative_weight_norm", "positive_n_seed_movies",
            "negative_n_seed_movies", "tag_ambiguity", "discriminative_lift",
            "discriminative_weight_raw", "discriminative_weight",
            "negative_discriminative_weight", "is_core_positive_tag", "is_core_negative_tag",
        ])

    for col in ["positive_weight", "negative_weight", "positive_n_seed_movies", "negative_n_seed_movies"]:
        profile[col] = pd.to_numeric(profile[col], errors="coerce").fillna(0.0)
    if "semantic_category" not in profile.columns:
        profile["semantic_category"] = None

    positive_total = profile["positive_weight"].sum()
    negative_total = profile["negative_weight"].sum()
    profile["positive_weight_norm"] = profile["positive_weight"] / positive_total if positive_total > 0 else 0.0
    profile["negative_weight_norm"] = profile["negative_weight"] / negative_total if negative_total > 0 else 0.0
    profile["tag_ambiguity"] = np.minimum(profile["positive_weight_norm"], profile["negative_weight_norm"])
    profile["discriminative_lift"] = np.log(
        (profile["positive_weight_norm"] + smoothing) / (profile["negative_weight_norm"] + smoothing)
    )
    profile["discriminative_weight_raw"] = (
        profile["positive_weight_norm"]
        * profile["discriminative_lift"].clip(lower=0.0)
        * (1 - profile["tag_ambiguity"])
    )

    many_positive_movies = len(positive_seed) >= 10
    profile.loc[many_positive_movies & (profile["positive_n_seed_movies"] == 1), "discriminative_weight_raw"] *= 0.70
    profile.loc[profile["positive_n_seed_movies"] >= 3, "discriminative_weight_raw"] *= 1.10
    profile.loc[profile["tag_ambiguity"] > 0.20, "discriminative_weight_raw"] *= 0.50
    profile.loc[profile["tag_ambiguity"] > 0.35, "discriminative_weight_raw"] *= 0.25
    profile.loc[profile["positive_n_seed_movies"] < min_positive_seed_movies, "discriminative_weight_raw"] = 0.0

    max_positive = profile["discriminative_weight_raw"].max()
    profile["discriminative_weight"] = profile["discriminative_weight_raw"] / max_positive if max_positive > 0 else 0.0

    profile["negative_discriminative_lift"] = np.log(
        (profile["negative_weight_norm"] + smoothing) / (profile["positive_weight_norm"] + smoothing)
    )
    profile["negative_discriminative_weight_raw"] = (
        profile["negative_weight_norm"]
        * profile["negative_discriminative_lift"].clip(lower=0.0)
        * (1 - profile["tag_ambiguity"])
    )
    max_negative = profile["negative_discriminative_weight_raw"].max()
    profile["negative_discriminative_weight"] = profile["negative_discriminative_weight_raw"] / max_negative if max_negative > 0 else 0.0

    profile["is_core_positive_tag"] = profile["discriminative_weight"] > 0
    profile["is_core_negative_tag"] = profile["negative_discriminative_weight"] > 0

    return profile[[
        "tag_clean", "semantic_category", "positive_weight", "negative_weight",
        "positive_weight_norm", "negative_weight_norm", "positive_n_seed_movies",
        "negative_n_seed_movies", "tag_ambiguity", "discriminative_lift",
        "discriminative_weight_raw", "discriminative_weight",
        "negative_discriminative_weight", "is_core_positive_tag", "is_core_negative_tag",
    ]].sort_values("discriminative_weight", ascending=False).reset_index(drop=True)


def compute_semantic_core_scores(candidates, tags_clean, discriminative_tag_profile):
    result = candidates[["movieId"]].copy()
    result["semantic_core_raw_score"] = 0.0
    result["semantic_core_negative_raw_score"] = 0.0
    result["semantic_core_score"] = 0.0
    result["semantic_core_negative_score"] = 0.0
    result["core_semantic_explanation_terms"] = ""
    result["core_negative_semantic_terms"] = ""

    if tags_clean is None or tags_clean.empty or discriminative_tag_profile is None or discriminative_tag_profile.empty:
        return result

    candidate_tags = tags_clean[tags_clean["movieId"].isin(result["movieId"].astype(int))][["movieId", "tag_clean"]].drop_duplicates()
    if candidate_tags.empty:
        return result

    profile_cols = ["tag_clean", "discriminative_weight", "negative_discriminative_weight"]
    weighted_tags = candidate_tags.merge(discriminative_tag_profile[profile_cols], on="tag_clean", how="inner")
    if weighted_tags.empty:
        return result

    positive_contrib = weighted_tags[weighted_tags["discriminative_weight"].fillna(0.0) > 0].copy()
    if not positive_contrib.empty:
        positive_contrib["contribution"] = positive_contrib["discriminative_weight"]
        raw_positive = positive_contrib.groupby("movieId")["contribution"].sum()
        result["semantic_core_raw_score"] = result["movieId"].map(raw_positive).fillna(0.0)
        result["semantic_core_score"] = _rank_positive(result["semantic_core_raw_score"])
        result["core_semantic_explanation_terms"] = result["movieId"].map(
            _top_terms_from_weighted_tags(positive_contrib, value_col="contribution", max_terms=5)
        ).fillna("")

    negative_contrib = weighted_tags[weighted_tags["negative_discriminative_weight"].fillna(0.0) > 0].copy()
    if not negative_contrib.empty:
        negative_contrib["contribution"] = negative_contrib["negative_discriminative_weight"]
        raw_negative = negative_contrib.groupby("movieId")["contribution"].sum()
        result["semantic_core_negative_raw_score"] = result["movieId"].map(raw_negative).fillna(0.0)
        result["semantic_core_negative_score"] = _rank_positive(result["semantic_core_negative_raw_score"])
        result["core_negative_semantic_terms"] = result["movieId"].map(
            _top_terms_from_weighted_tags(negative_contrib, value_col="contribution", max_terms=5)
        ).fillna("")

    return result


def compute_semantic_net_score(df, beta=0.75):
    scored = df.copy()
    if {"semantic_core_raw_score", "semantic_core_negative_raw_score"}.issubset(scored.columns):
        semantic_raw = scored["semantic_core_raw_score"]
        negative_raw = scored["semantic_core_negative_raw_score"]
    else:
        semantic_raw = scored["semantic_raw_score"] if "semantic_raw_score" in scored.columns else scored.get("semantic_profile_score", 0.0)
        negative_raw = scored["negative_semantic_raw_score"] if "negative_semantic_raw_score" in scored.columns else scored.get("negative_semantic_score", 0.0)
    scored["semantic_net_raw_score"] = semantic_raw.fillna(0.0) - beta * negative_raw.fillna(0.0)
    scored["semantic_net_score"] = _rank_positive(scored["semantic_net_raw_score"])
    return scored


def _first_weight_column(df, preferred):
    if df is None or df.empty:
        return None
    for col in preferred:
        if col in df.columns:
            return col
    numeric_cols = [col for col in df.select_dtypes(include=[np.number]).columns if not col.lower().endswith("id")]
    return numeric_cols[0] if numeric_cols else None


def _normalized_tag_weights(df, preferred_weight_cols):
    if df is None or df.empty or "tag_clean" not in df.columns:
        return pd.DataFrame(columns=["tag_clean", "weight_norm"])
    weight_col = _first_weight_column(df, preferred_weight_cols)
    if weight_col is None:
        return pd.DataFrame(columns=["tag_clean", "weight_norm"])
    weights = df[["tag_clean", weight_col]].dropna().copy()
    weights[weight_col] = pd.to_numeric(weights[weight_col], errors="coerce").fillna(0.0).clip(lower=0.0)
    weights = weights.groupby("tag_clean", as_index=False)[weight_col].sum()
    total = weights[weight_col].sum()
    if total <= 0:
        return pd.DataFrame(columns=["tag_clean", "weight_norm"])
    weights["weight_norm"] = weights[weight_col] / total
    return weights[["tag_clean", "weight_norm"]]


def _semantic_ambiguity(positive_tag_profile, negative_tag_profile):
    positive_weights = _normalized_tag_weights(positive_tag_profile, ["positive_weight", "tag_weight", "weight"])
    negative_weights = _normalized_tag_weights(negative_tag_profile, ["negative_weight", "tag_weight", "weight"])
    if positive_weights.empty or negative_weights.empty:
        return 1.0, 0
    overlap = positive_weights.merge(negative_weights, on="tag_clean", how="inner", suffixes=("_pos", "_neg"))
    if overlap.empty:
        return 0.0, 0
    ambiguity = np.minimum(overlap["weight_norm_pos"], overlap["weight_norm_neg"]).sum()
    return _clip01(ambiguity), len(overlap)


def extract_genre_weights_from_profile(df, rating_col="user_rating_5", positive=True):
    if df is None or df.empty or "genres" not in df.columns or rating_col not in df.columns:
        return pd.DataFrame(columns=["genre", "genre_weight_norm"])
    rows = []
    for _, row in df.dropna(subset=[rating_col]).iterrows():
        base_weight = row[rating_col] - 3.0 if positive else 3.0 - row[rating_col]
        if base_weight <= 0:
            continue
        for genre in split_genres(row.get("genres", "")):
            rows.append({"genre": genre, "weight": base_weight})
    if not rows:
        return pd.DataFrame(columns=["genre", "genre_weight_norm"])
    weights = pd.DataFrame(rows).groupby("genre", as_index=False)["weight"].sum()
    total = weights["weight"].sum()
    if total <= 0:
        return pd.DataFrame(columns=["genre", "genre_weight_norm"])
    weights["genre_weight_norm"] = weights["weight"] / total
    return weights[["genre", "genre_weight_norm"]]



def _valid_year_series(df, year_col="year"):
    if df is None or df.empty or year_col not in df.columns:
        return pd.Series(dtype=float)
    years = pd.to_numeric(df[year_col], errors="coerce")
    return years[(years >= 1874) & (years <= 2035)].dropna()


def analyze_temporal_profile(
    user_movies,
    liked_movies=None,
    watched_movies=None,
    year_col="year",
    rating_col="user_rating_5",
):
    liked_years = _valid_year_series(liked_movies, year_col)
    user_years = _valid_year_series(user_movies, year_col)
    watched_years = _valid_year_series(watched_movies, year_col)

    profile_years = liked_years if len(liked_years) >= 5 else user_years
    if len(profile_years) < 5 and len(watched_years) > len(profile_years):
        profile_years = watched_years

    liked_bounds_years = liked_years if not liked_years.empty else profile_years

    if profile_years.empty:
        return {
            "n_temporal_movies": 0,
            "min_year": np.nan,
            "max_year": np.nan,
            "q10_year": np.nan,
            "q25_year": np.nan,
            "median_year": np.nan,
            "q75_year": np.nan,
            "q90_year": np.nan,
            "mean_year": np.nan,
            "std_year": np.nan,
            "earliest_liked_year": liked_bounds_years.min() if not liked_bounds_years.empty else np.nan,
            "latest_liked_year": liked_bounds_years.max() if not liked_bounds_years.empty else np.nan,
            "pct_before_1960": 0.0,
            "pct_before_1970": 0.0,
            "pct_before_1980": 0.0,
            "pct_before_1990": 0.0,
            "pct_after_2000": 0.0,
            "pct_after_2010": 0.0,
            "temporal_profile_strength": 0.0,
            "classic_tolerance": 0.0,
            "temporal_strictness": 0.0,
        }

    n_temporal_movies = len(profile_years)
    q10_year = float(profile_years.quantile(0.10))
    q90_year = float(profile_years.quantile(0.90))
    pct_before_1960 = float((profile_years < 1960).mean())
    pct_before_1970 = float((profile_years < 1970).mean())
    pct_before_1980 = float((profile_years < 1980).mean())
    pct_before_1990 = float((profile_years < 1990).mean())
    pct_after_2000 = float((profile_years >= 2000).mean())
    pct_after_2010 = float((profile_years >= 2010).mean())

    if n_temporal_movies < 5:
        temporal_profile_strength = min(0.4, n_temporal_movies / 10)
    else:
        temporal_profile_strength = _clip01(
            min(1.0, n_temporal_movies / 30) * 0.60
            + min(1.0, max(1.0, q90_year - q10_year) / 40) * 0.10
            + 0.30
        )

    classic_tolerance = _clip01(
        0.50 * pct_before_1980
        + 0.30 * pct_before_1970
        + 0.20 * pct_before_1960
    )
    temporal_strictness = _clip01(temporal_profile_strength * (1 - classic_tolerance))

    return {
        "n_temporal_movies": int(n_temporal_movies),
        "min_year": float(profile_years.min()),
        "max_year": float(profile_years.max()),
        "q10_year": q10_year,
        "q25_year": float(profile_years.quantile(0.25)),
        "median_year": float(profile_years.median()),
        "q75_year": float(profile_years.quantile(0.75)),
        "q90_year": q90_year,
        "mean_year": float(profile_years.mean()),
        "std_year": float(profile_years.std(ddof=0)) if n_temporal_movies > 1 else 0.0,
        "earliest_liked_year": float(liked_bounds_years.min()) if not liked_bounds_years.empty else np.nan,
        "latest_liked_year": float(liked_bounds_years.max()) if not liked_bounds_years.empty else np.nan,
        "pct_before_1960": pct_before_1960,
        "pct_before_1970": pct_before_1970,
        "pct_before_1980": pct_before_1980,
        "pct_before_1990": pct_before_1990,
        "pct_after_2000": pct_after_2000,
        "pct_after_2010": pct_after_2010,
        "temporal_profile_strength": temporal_profile_strength,
        "classic_tolerance": classic_tolerance,
        "temporal_strictness": temporal_strictness,
    }


def compute_year_affinity_scores(candidates, temporal_metrics, year_col="year"):
    scored = candidates.copy()
    temporal_profile_strength = float(temporal_metrics.get("temporal_profile_strength", 0.0) or 0.0)
    temporal_strictness = float(temporal_metrics.get("temporal_strictness", 0.0) or 0.0)
    classic_tolerance = float(temporal_metrics.get("classic_tolerance", 0.0) or 0.0)

    scored["year_affinity_score"] = 1.0
    scored["temporal_mismatch_penalty"] = 0.0
    scored["temporal_distance_from_profile"] = 0.0
    scored["is_temporal_outlier"] = False

    years = pd.to_numeric(scored.get(year_col), errors="coerce")
    invalid_year_mask = years.isna()
    if invalid_year_mask.any():
        scored.loc[invalid_year_mask, "year_affinity_score"] = 0.5
        scored.loc[invalid_year_mask, "temporal_mismatch_penalty"] = 0.5 * temporal_strictness
        scored.loc[invalid_year_mask, "temporal_distance_from_profile"] = np.nan
        scored.loc[invalid_year_mask, "is_temporal_outlier"] = True

    if temporal_profile_strength < 0.25:
        scored.loc[~invalid_year_mask, "year_affinity_score"] = 1.0
        scored.loc[~invalid_year_mask, "temporal_mismatch_penalty"] = 0.0
        scored.loc[~invalid_year_mask, "temporal_distance_from_profile"] = 0.0
        scored.loc[~invalid_year_mask, "is_temporal_outlier"] = False
        return scored

    q10_year = temporal_metrics.get("q10_year")
    q90_year = temporal_metrics.get("q90_year")
    if pd.isna(q10_year) or pd.isna(q90_year):
        return scored

    preferred_low = float(q10_year) - 5
    preferred_high = float(q90_year) + 5
    valid_year_mask = ~invalid_year_mask
    before_mask = valid_year_mask & (years < preferred_low)
    after_mask = valid_year_mask & (years > preferred_high)
    inside_mask = valid_year_mask & ~(before_mask | after_mask)

    scored.loc[inside_mask, "year_affinity_score"] = 1.0
    scored.loc[inside_mask, "temporal_distance_from_profile"] = 0.0

    if after_mask.any():
        distance = years.loc[after_mask] - preferred_high
        scored.loc[after_mask, "year_affinity_score"] = np.maximum(np.exp(-distance / 35), 0.65)
        scored.loc[after_mask, "temporal_distance_from_profile"] = distance

    if before_mask.any():
        distance = preferred_low - years.loc[before_mask]
        decay = 15 + 35 * classic_tolerance
        floor = 0.35 if classic_tolerance >= 0.30 else 0.05
        scored.loc[before_mask, "year_affinity_score"] = np.maximum(np.exp(-distance / decay), floor)
        scored.loc[before_mask, "temporal_distance_from_profile"] = distance

    scored["year_affinity_score"] = scored["year_affinity_score"].clip(0.0, 1.0)
    scored["temporal_mismatch_penalty"] = (1 - scored["year_affinity_score"]) * temporal_strictness
    scored.loc[invalid_year_mask, "temporal_mismatch_penalty"] = 0.5 * temporal_strictness
    scored["is_temporal_outlier"] = (
        (years < preferred_low - 20)
        & (classic_tolerance < 0.20)
        & (temporal_profile_strength >= 0.50)
    ).fillna(False)
    scored.loc[invalid_year_mask, "is_temporal_outlier"] = True
    return scored


def apply_temporal_outlier_allowance(candidates, min_candidates_after_filter=100):
    scored = candidates.copy()
    if "is_temporal_outlier" not in scored.columns:
        scored["allow_temporal_outlier"] = True
        return scored

    collab_p80 = scored["item_item_collab_score"].quantile(0.80) if "item_item_collab_score" in scored.columns else np.inf
    semantic_reference = "semantic_net_score" if "semantic_net_score" in scored.columns else "semantic_profile_score"
    semantic_p90 = scored[semantic_reference].quantile(0.90) if semantic_reference in scored.columns else np.inf
    rating_p95 = scored["rating_score"].quantile(0.95) if "rating_score" in scored.columns else np.inf

    scored["allow_temporal_outlier"] = (
        ~scored["is_temporal_outlier"].fillna(False)
        | (scored.get("item_item_collab_score", 0) >= collab_p80)
        | (scored.get(semantic_reference, 0) >= semantic_p90)
        | (scored.get("rating_score", 0) >= rating_p95)
    )

    filtered = scored[scored["allow_temporal_outlier"]].copy()
    if len(filtered) >= min_candidates_after_filter:
        return filtered

    warnings.warn(
        "No se aplica el filtro suave de outliers temporales porque dejar?a menos de "
        f"{min_candidates_after_filter} candidatos."
    )
    scored["allow_temporal_outlier"] = True
    return scored

def analyze_user_profile_for_adaptive_weights(
    liked_movies,
    neutral_movies,
    disliked_movies,
    positive_tag_profile,
    negative_tag_profile,
    trakt_ratings_profile,
    movie_id_to_col=None,
    candidates_scored=None,
    discriminative_tag_profile=None,
):
    n_liked = len(liked_movies)
    n_neutral = len(neutral_movies)
    n_disliked = len(disliked_movies)
    n_total_ratings = len(trakt_ratings_profile)
    profile_size_score = _clip01(n_total_ratings / 80)
    liked_size_score = _clip01(n_liked / 30)
    disliked_size_score = _clip01(n_disliked / 15)

    n_positive_semantic_tags = len(positive_tag_profile) if positive_tag_profile is not None else 0
    n_negative_semantic_tags = len(negative_tag_profile) if negative_tag_profile is not None else 0
    semantic_ambiguity, n_shared_semantic_tags = _semantic_ambiguity(positive_tag_profile, negative_tag_profile)

    if positive_tag_profile is not None and "semantic_category" in positive_tag_profile.columns:
        category_strength = _clip01(positive_tag_profile["semantic_category"].dropna().nunique() / 4)
    else:
        category_strength = 0.5
    semantic_profile_strength = _clip01(
        0.45 * min(1.0, n_positive_semantic_tags / 40)
        + 0.35 * liked_size_score
        + 0.20 * category_strength
    )
    semantic_reliability = _clip01(semantic_profile_strength * (1 - semantic_ambiguity))

    n_core_positive_tags = 0
    n_core_negative_tags = 0
    mean_core_tag_ambiguity = semantic_ambiguity
    core_semantic_strength = semantic_profile_strength
    if discriminative_tag_profile is not None and not discriminative_tag_profile.empty:
        core_positive = discriminative_tag_profile[discriminative_tag_profile["is_core_positive_tag"]].copy()
        core_negative = discriminative_tag_profile[discriminative_tag_profile["is_core_negative_tag"]].copy()
        n_core_positive_tags = len(core_positive)
        n_core_negative_tags = len(core_negative)
        mean_core_tag_ambiguity = core_positive["tag_ambiguity"].mean() if not core_positive.empty else 1.0
        if not core_positive.empty and "semantic_category" in core_positive.columns:
            n_core_categories = core_positive["semantic_category"].dropna().nunique()
        else:
            n_core_categories = 0
        core_semantic_strength = _clip01(
            min(1.0, n_core_positive_tags / 30) * 0.45
            + min(1.0, n_core_categories / 4) * 0.25
            + liked_size_score * 0.30
        )
        semantic_reliability = _clip01(core_semantic_strength * (1 - mean_core_tag_ambiguity))

    negative_profile_strength = _clip01(0.50 * disliked_size_score + 0.50 * min(1.0, max(n_negative_semantic_tags, n_core_negative_tags) / 30))
    negative_semantic_reliability = _clip01(negative_profile_strength * (1 - mean_core_tag_ambiguity))

    positive_genres = extract_genre_weights_from_profile(liked_movies, positive=True)
    negative_genres = extract_genre_weights_from_profile(disliked_movies, positive=False)
    n_positive_genres = len(positive_genres)
    n_negative_genres = len(negative_genres)
    if positive_genres.empty or negative_genres.empty:
        genre_ambiguity = 1.0
    else:
        genre_overlap = positive_genres.merge(negative_genres, on="genre", how="inner", suffixes=("_pos", "_neg"))
        genre_ambiguity = _clip01(np.minimum(genre_overlap["genre_weight_norm_pos"], genre_overlap["genre_weight_norm_neg"]).sum()) if not genre_overlap.empty else 0.0
    genre_profile_strength = _clip01(0.50 * min(1.0, n_positive_genres / 6) + 0.50 * liked_size_score)
    genre_reliability = _clip01(genre_profile_strength * (1 - genre_ambiguity))

    if movie_id_to_col is not None:
        rated_ids = set(trakt_ratings_profile["movieId"].dropna().astype(int)) if "movieId" in trakt_ratings_profile.columns else set()
        collab_coverage = len(rated_ids & set(movie_id_to_col.keys())) / len(rated_ids) if rated_ids else 0.0
    else:
        collab_coverage = 0.5
    if candidates_scored is not None and "n_collab_evidence" in candidates_scored.columns and len(candidates_scored):
        collab_candidate_evidence = (candidates_scored["n_collab_evidence"].fillna(0) > 0).mean()
        collab_reliability = _clip01(0.70 * collab_coverage + 0.30 * collab_candidate_evidence)
    else:
        collab_candidate_evidence = np.nan
        collab_reliability = _clip01(collab_coverage)

    profile_weakness = _clip01(1 - profile_size_score)
    return {
        "n_liked": n_liked,
        "n_neutral": n_neutral,
        "n_disliked": n_disliked,
        "n_total_ratings": n_total_ratings,
        "profile_size_score": profile_size_score,
        "liked_size_score": liked_size_score,
        "disliked_size_score": disliked_size_score,
        "n_positive_semantic_tags": n_positive_semantic_tags,
        "n_negative_semantic_tags": n_negative_semantic_tags,
        "n_shared_semantic_tags": n_shared_semantic_tags,
        "semantic_ambiguity": semantic_ambiguity,
        "semantic_profile_strength": semantic_profile_strength,
        "semantic_reliability": semantic_reliability,
        "negative_profile_strength": negative_profile_strength,
        "negative_semantic_reliability": negative_semantic_reliability,
        "n_core_positive_tags": n_core_positive_tags,
        "n_core_negative_tags": n_core_negative_tags,
        "mean_core_tag_ambiguity": _clip01(mean_core_tag_ambiguity),
        "core_semantic_strength": core_semantic_strength,
        "n_positive_genres": n_positive_genres,
        "n_negative_genres": n_negative_genres,
        "genre_ambiguity": genre_ambiguity,
        "genre_profile_strength": genre_profile_strength,
        "genre_reliability": genre_reliability,
        "collab_coverage": _clip01(collab_coverage),
        "collab_candidate_evidence": collab_candidate_evidence,
        "collab_reliability": collab_reliability,
        "profile_weakness": profile_weakness,
    }


def _bounded_normalize(raw_weights, target_total, min_bounds=None, max_bounds=None):
    min_bounds = min_bounds or {}
    max_bounds = max_bounds or {}
    keys = list(raw_weights)
    weights = {key: max(float(raw_weights[key]), 0.0) for key in keys}
    raw_total = sum(weights.values())
    if raw_total <= 0:
        return {key: target_total / len(keys) for key in keys}
    weights = {key: value / raw_total * target_total for key, value in weights.items()}
    fixed = {}
    free = set(keys)
    for _ in range(10):
        changed = False
        for key in list(free):
            lower = min_bounds.get(key, 0.0)
            upper = max_bounds.get(key, float("inf"))
            if weights[key] < lower:
                fixed[key] = lower
                free.remove(key)
                changed = True
            elif weights[key] > upper:
                fixed[key] = upper
                free.remove(key)
                changed = True
        remaining = target_total - sum(fixed.values())
        if remaining <= 0 or not free:
            break
        free_raw_total = sum(raw_weights[key] for key in free)
        if free_raw_total <= 0:
            for key in free:
                weights[key] = remaining / len(free)
        else:
            for key in free:
                weights[key] = raw_weights[key] / free_raw_total * remaining
        if not changed:
            break
    weights.update(fixed)
    return weights


def derive_adaptive_weights(profile_metrics):
    genre_reliability = profile_metrics.get("genre_reliability", 0.0)
    semantic_reliability = profile_metrics.get("semantic_reliability", 0.0)
    collab_reliability = profile_metrics.get("collab_reliability", 0.0)
    negative_semantic_reliability = profile_metrics.get("negative_semantic_reliability", 0.0)
    negative_profile_strength = profile_metrics.get("negative_profile_strength", 0.0)
    profile_weakness = profile_metrics.get("profile_weakness", 1.0)
    temporal_profile_strength = profile_metrics.get("temporal_profile_strength", 0.0)
    temporal_strictness = profile_metrics.get("temporal_strictness", 0.0)

    raw_positive = {
        "genre": 0.06 + 0.12 * genre_reliability,
        "semantic": 0.10 + 0.35 * semantic_reliability,
        "collab": 0.10 + 0.22 * collab_reliability,
        "rating": 0.14 + 0.14 * profile_weakness,
        "popularity": 0.02 + 0.08 * profile_weakness,
        "year_affinity": 0.04 + 0.08 * temporal_profile_strength,
    }
    raw_negative = {
        "negative_genre": 0.10 + 0.15 * genre_reliability * negative_profile_strength,
        "negative_semantic": 0.15 + 0.35 * negative_semantic_reliability,
        "negative_collab": 0.08 + 0.20 * collab_reliability * negative_profile_strength,
        "temporal_mismatch": 0.08 + 0.22 * temporal_strictness,
    }

    positive_min = {"rating": 0.12}
    if collab_reliability >= 0.70:
        positive_min["collab"] = 0.18
    positive_max = {"semantic": 0.30, "popularity": 0.06, "collab": 0.25, "rating": 0.25, "year_affinity": 0.10}
    negative_max = {"negative_semantic": 0.35, "temporal_mismatch": 0.22}

    weights = _bounded_normalize(raw_positive, 0.75, min_bounds=positive_min, max_bounds=positive_max)
    weights.update(_bounded_normalize(raw_negative, 0.60, max_bounds=negative_max))
    return weights


discriminative_tag_profile = build_discriminative_semantic_profile(
    tags_clean=tags_clean,
    liked_movies=liked_movies,
    disliked_movies=disliked_movies,
    positive_tag_profile=positive_tag_profile,
    negative_tag_profile=negative_tag_profile,
)

print("Top 30 tags por discriminative_weight")
display(discriminative_tag_profile.sort_values("discriminative_weight", ascending=False).head(30))
print("Top 30 tags por negative_discriminative_weight")
display(discriminative_tag_profile.sort_values("negative_discriminative_weight", ascending=False).head(30))
print("Media de tag_ambiguity:", discriminative_tag_profile["tag_ambiguity"].mean() if not discriminative_tag_profile.empty else np.nan)
print("Número de core positive tags:", int(discriminative_tag_profile["is_core_positive_tag"].sum()) if not discriminative_tag_profile.empty else 0)
print("Número de core negative tags:", int(discriminative_tag_profile["is_core_negative_tag"].sum()) if not discriminative_tag_profile.empty else 0)
if not discriminative_tag_profile.empty and "semantic_category" in discriminative_tag_profile.columns:
    display(discriminative_tag_profile.loc[discriminative_tag_profile["is_core_positive_tag"], "semantic_category"].value_counts(dropna=False))

discriminative_tag_profile.to_csv(REPORTS_RESULTADOS / "discriminative_tag_profile.csv", index=False)

semantic_core_scores = compute_semantic_core_scores(
    candidates=candidates_scored,
    tags_clean=tags_clean,
    discriminative_tag_profile=discriminative_tag_profile,
)
candidates_scored = candidates_scored.drop(columns=[col for col in semantic_core_scores.columns if col != "movieId" and col in candidates_scored.columns]).merge(
    semantic_core_scores,
    on="movieId",
    how="left",
)
for col in ["semantic_core_raw_score", "semantic_core_negative_raw_score", "semantic_core_score", "semantic_core_negative_score"]:
    candidates_scored[col] = candidates_scored[col].fillna(0.0)
for col in ["core_semantic_explanation_terms", "core_negative_semantic_terms"]:
    candidates_scored[col] = candidates_scored[col].fillna("")


temporal_metrics = analyze_temporal_profile(
    user_movies=trakt_ratings_profile,
    liked_movies=liked_movies,
    watched_movies=trakt_watched.merge(movies[["movieId", "year"]], on="movieId", how="left") if "year" not in trakt_watched.columns else trakt_watched,
)
temporal_metrics_df = pd.DataFrame(temporal_metrics.items(), columns=["metric", "value"])
display(temporal_metrics_df)
temporal_metrics_df.to_csv(REPORTS_RESULTADOS / "temporal_profile_metrics.csv", index=False)

candidates_scored = compute_year_affinity_scores(candidates_scored, temporal_metrics, year_col="year")

profile_metrics = analyze_user_profile_for_adaptive_weights(
    liked_movies=liked_movies,
    neutral_movies=neutral_movies,
    disliked_movies=disliked_movies,
    positive_tag_profile=positive_tag_profile,
    negative_tag_profile=negative_tag_profile,
    trakt_ratings_profile=trakt_ratings,
    movie_id_to_col=movie_id_to_col,
    candidates_scored=candidates_scored,
    discriminative_tag_profile=discriminative_tag_profile,
)
profile_metrics.update(temporal_metrics)
profile_metrics_df = pd.DataFrame(profile_metrics.items(), columns=["metric", "value"])
display(profile_metrics_df.sort_values("metric"))
profile_metrics_df.to_csv(REPORTS_RESULTADOS / "adaptive_profile_metrics.csv", index=False)

semantic_negative_beta = float(np.clip(0.50 + 0.75 * profile_metrics["negative_profile_strength"], 0.50, 1.25))
candidates_scored = compute_semantic_net_score(candidates_scored, beta=semantic_negative_beta)
temporal_outlier_candidate_count = int(candidates_scored["is_temporal_outlier"].sum()) if "is_temporal_outlier" in candidates_scored.columns else 0
candidates_scored = apply_temporal_outlier_allowance(candidates_scored)
temporal_outlier_allowed_count = int((candidates_scored["is_temporal_outlier"] & candidates_scored["allow_temporal_outlier"]).sum()) if {"is_temporal_outlier", "allow_temporal_outlier"}.issubset(candidates_scored.columns) else 0
temporal_outlier_removed_count = max(0, temporal_outlier_candidate_count - temporal_outlier_allowed_count)
candidates_scored["semantic_negative_beta"] = semantic_negative_beta

adaptive_weights = derive_adaptive_weights(profile_metrics)
adaptive_weights_df = pd.DataFrame(adaptive_weights.items(), columns=["component", "weight"])
display(adaptive_weights_df)
adaptive_weights_df.to_csv(REPORTS_RESULTADOS / "adaptive_weights.csv", index=False)

print(f"semantic_negative_beta: {semantic_negative_beta:.3f}")
print("Pesos adaptativos calculados:")
print(adaptive_weights)

Top 30 tags por discriminative_weight


,tag_clean,semantic_category,positive_weight,negative_weight,positive_weight_norm,negative_weight_norm,positive_n_seed_movies,negative_n_seed_movies,tag_ambiguity,discriminative_lift,discriminative_weight_raw,discriminative_weight,negative_discriminative_weight,is_core_positive_tag,is_core_negative_tag
0,tense,tone_atmosphere,64.307744,0.000000,0.010534,0.000000,12.0,0.0,0.000000,0.100153,0.001161,1.000000,0.0,True,False
1,dark,tone_atmosphere,73.126727,2.358927,0.011979,0.002606,14.0,1.0,0.002606,0.087415,0.001149,0.989921,0.0,True,False
2,psychology,theme,54.468484,0.000000,0.008922,0.000000,10.0,0.0,0.000000,0.085465,0.000839,0.722775,0.0,True,False
3,cinematography,visual_style,104.322034,10.866879,0.017089,0.012004,21.0,3.0,0.012004,0.044401,0.000825,0.710552,0.0,True,False
4,nostalgic,tone_atmosphere,53.348209,0.000000,0.008739,0.000000,7.0,0.0,0.000000,0.083778,0.000805,0.693942,0.0,True,False
5,existentialism,theme,52.521941,0.000000,0.008603,0.000000,8.0,0.0,0.000000,0.082533,0.000781,0.673038,0.0,True,False
6,ambiguous ending,narrative_structure,52.335802,0.000000,0.008573,0.000000,6.0,0.0,0.000000,0.082252,0.000776,0.668371,0.0,True,False
7,dreamlike,visual_style,50.012098,0.000000,0.008192,0.000000,8.0,0.0,0.000000,0.078740,0.000710,0.611424,0.0,True,False
8,surrealism,visual_style,49.732575,0.000000,0.008147,0.000000,10.0,0.0,0.000000,0.078317,0.000702,0.604738,0.0,True,False
9,atmospheric,tone_atmosphere,111.034884,12.811717,0.018188,0.014152,22.0,4.0,0.014152,0.034750,0.000685,0.590599,0.0,True,False


Top 30 tags por negative_discriminative_weight


,tag_clean,semantic_category,positive_weight,negative_weight,positive_weight_norm,negative_weight_norm,positive_n_seed_movies,negative_n_seed_movies,tag_ambiguity,discriminative_lift,discriminative_weight_raw,discriminative_weight,negative_discriminative_weight,is_core_positive_tag,is_core_negative_tag
210,long takes,NaN,0.000000,22.442782,0.000000,0.024790,0.0,2.0,0.000000,-0.221464,0.0,0.0,1.000000,False,True
277,surprise ending,narrative_structure,67.747161,24.039315,0.011097,0.026554,14.0,5.0,0.011097,-0.130259,0.0,0.0,0.623025,False,True
204,twist ending,narrative_structure,60.228121,22.585545,0.009866,0.024948,9.0,3.0,0.009866,-0.128638,0.0,0.0,0.578780,False,True
223,robots,theme,17.900399,17.900399,0.002932,0.019773,3.0,3.0,0.002932,-0.151525,0.0,0.0,0.544118,False,True
211,visually appealing,visual_style,79.191596,23.997453,0.012972,0.026507,14.0,6.0,0.012972,-0.113161,0.0,0.0,0.539275,False,True
272,technology,theme,16.684003,16.684003,0.002733,0.018429,3.0,3.0,0.002733,-0.142181,0.0,0.0,0.475965,False,True
267,dark humor,tone_atmosphere,22.942751,17.207063,0.003758,0.019007,3.0,3.0,0.003758,-0.137118,0.0,0.0,0.472919,False,True
260,shaky camera,NaN,0.000000,14.091218,0.000000,0.015565,0.0,1.0,0.000000,-0.144664,0.0,0.0,0.410137,False,True
276,stupid ending,narrative_structure,7.170772,14.341545,0.001175,0.015842,1.0,1.0,0.001175,-0.135376,0.0,0.0,0.390164,False,True
212,unexpected ending,narrative_structure,14.386490,14.386490,0.002357,0.015891,2.0,1.0,0.002357,-0.124189,0.0,0.0,0.358621,False,True


Media de tag_ambiguity: 0.0014676250137088621
Número de core positive tags: 199
Número de core negative tags: 88


semantic_category
theme                             71
narrative_structure               37
tone_atmosphere                   29
visual_style                      17
humor                             17
emotional_cognitive_experience    14
intensity_darkness                14
Name: count, dtype: int64

,metric,value
0,n_temporal_movies,218.000000
1,min_year,1968.000000
2,max_year,2023.000000
3,q10_year,1985.400000
4,q25_year,1999.000000
5,median_year,2008.500000
6,q75_year,2015.000000
7,q90_year,2021.000000
8,mean_year,2005.807339
9,std_year,12.816616


,metric,value
47,classic_tolerance,0.031193
25,collab_candidate_evidence,0.412503
24,collab_coverage,0.967480
26,collab_reliability,0.800987
18,core_semantic_strength,1.000000
6,disliked_size_score,1.000000
38,earliest_liked_year,1968.000000
21,genre_ambiguity,0.699269
22,genre_profile_strength,1.000000
23,genre_reliability,0.300731


,component,weight
0,genre,0.061278
1,semantic,0.300000
2,collab,0.180000
3,rating,0.120000
4,popularity,0.012755
5,year_affinity,0.075967
6,negative_genre,0.074043
7,negative_semantic,0.255017
8,negative_collab,0.122562
9,temporal_mismatch,0.148379


semantic_negative_beta: 1.250
Pesos adaptativos calculados:
{'genre': 0.061278422059187716, 'semantic': 0.3, 'collab': 0.18, 'rating': 0.12, 'popularity': 0.01275468343024904, 'year_affinity': 0.07596689451056327, 'negative_genre': 0.07404290840570853, 'negative_semantic': 0.2550166372476469, 'negative_collab': 0.12256185191463556, 'temporal_mismatch': 0.148378602432009}


## Análisis y configuración de pesos del score híbrido

In [16]:
WEIGHT_CONFIGS = {
    "personalizado_semantico": {
        "genre": 0.18,
        "semantic": 0.35,
        "collab": 0.12,
        "rating": 0.15,
        "popularity": 0.03,
        "negative_genre": 0.20,
        "negative_semantic": 0.30,
        "negative_collab": 0.15,
    },
    "equilibrado_semantico": {
        "genre": 0.20,
        "semantic": 0.30,
        "collab": 0.15,
        "rating": 0.17,
        "popularity": 0.04,
        "negative_genre": 0.20,
        "negative_semantic": 0.25,
        "negative_collab": 0.15,
    },
    "descubrimiento_semantico": {
        "genre": 0.18,
        "semantic": 0.40,
        "collab": 0.08,
        "rating": 0.14,
        "popularity": 0.00,
        "negative_genre": 0.20,
        "negative_semantic": 0.35,
        "negative_collab": 0.20,
    },
}

ACTIVE_WEIGHT_CONFIG = "adaptive"


def compute_hybrid_score(df, weights):
    scored = df.copy()
    scored["contrib_genre"] = weights["genre"] * scored["genre_profile_score"]
    if "year_affinity_score" not in scored.columns:
        scored["year_affinity_score"] = 1.0
    if "temporal_mismatch_penalty" not in scored.columns:
        scored["temporal_mismatch_penalty"] = 0.0
    if "semantic_net_score" in scored.columns:
        semantic_signal = scored["semantic_net_score"]
    elif "semantic_core_score" in scored.columns:
        semantic_signal = scored["semantic_core_score"]
    else:
        semantic_signal = scored["semantic_profile_score"]
    scored["contrib_semantic"] = weights["semantic"] * semantic_signal
    scored["contrib_collab"] = weights["collab"] * scored["item_item_collab_score"]
    scored["contrib_rating"] = weights["rating"] * scored["rating_score"]
    scored["contrib_popularity"] = weights["popularity"] * scored["popularity_score"]
    scored["contrib_year_affinity"] = weights.get("year_affinity", 0.0) * scored["year_affinity_score"]
    scored["penalty_negative_genre"] = weights["negative_genre"] * scored["negative_genre_score"]
    scored["penalty_negative_semantic"] = weights["negative_semantic"] * scored["negative_semantic_score"]
    scored["penalty_negative_collab"] = weights["negative_collab"] * scored["item_item_negative_collab_score"]
    scored["penalty_temporal_mismatch"] = weights.get("temporal_mismatch", 0.0) * scored["temporal_mismatch_penalty"]

    scored["hybrid_score"] = (
        scored["contrib_genre"]
        + scored["contrib_semantic"]
        + scored["contrib_collab"]
        + scored["contrib_rating"]
        + scored["contrib_popularity"]
        + scored["contrib_year_affinity"]
        - scored["penalty_negative_genre"]
        - scored["penalty_negative_semantic"]
        - scored["penalty_negative_collab"]
        - scored["penalty_temporal_mismatch"]
    )

    contribution_cols = {
        "semantic": "contrib_semantic",
        "genre": "contrib_genre",
        "collab": "contrib_collab",
        "rating": "contrib_rating",
        "popularity": "contrib_popularity",
        "year_affinity": "contrib_year_affinity",
    }
    scored["dominant_signal"] = scored[list(contribution_cols.values())].idxmax(axis=1)
    scored["dominant_signal"] = scored["dominant_signal"].replace({value: key for key, value in contribution_cols.items()})
    return scored


candidates_scored_base = candidates_scored.copy()
print(f"Configuración activa de pesos: {ACTIVE_WEIGHT_CONFIG}")

Configuración activa de pesos: adaptive


## 14. Hybrid score inicial

In [17]:
weights = adaptive_weights if ACTIVE_WEIGHT_CONFIG == "adaptive" else WEIGHT_CONFIGS[ACTIVE_WEIGHT_CONFIG]
candidates_scored = compute_hybrid_score(candidates_scored, weights)

candidates_scored = candidates_scored.sort_values("hybrid_score", ascending=False).reset_index(drop=True)
candidates_scored["hybrid_rank_raw"] = np.arange(1, len(candidates_scored) + 1)

AUDIT_COLUMNS = [
    "title",
    "genres",
    "genre_profile_score",
    "semantic_profile_score",
    "semantic_core_score",
    "semantic_core_negative_score",
    "semantic_net_score",
    "negative_genre_score",
    "negative_semantic_score",
    "semantic_negative_beta",
    "item_item_collab_score",
    "item_item_negative_collab_score",
    "rating_score",
    "popularity_score",
    "allow_temporal_outlier",
    "is_temporal_outlier",
    "temporal_distance_from_profile",
    "temporal_mismatch_penalty",
    "year_affinity_score",
    "contrib_genre",
    "contrib_semantic",
    "contrib_collab",
    "contrib_rating",
    "contrib_popularity",
    "contrib_year_affinity",
    "penalty_negative_genre",
    "penalty_negative_semantic",
    "penalty_negative_collab",
    "penalty_temporal_mismatch",
    "dominant_signal",
    "semantic_explanation_terms",
    "core_semantic_explanation_terms",
    "core_negative_semantic_terms",
    "hybrid_score",
]

display(candidates_scored[AUDIT_COLUMNS].head(20))
display(candidates_scored.head(50)["dominant_signal"].value_counts())

,title,genres,genre_profile_score,semantic_profile_score,semantic_core_score,semantic_core_negative_score,semantic_net_score,negative_genre_score,negative_semantic_score,semantic_negative_beta,...,contrib_year_affinity,penalty_negative_genre,penalty_negative_semantic,penalty_negative_collab,penalty_temporal_mismatch,dominant_signal,semantic_explanation_terms,core_semantic_explanation_terms,core_negative_semantic_terms,hybrid_score
0,25th Hour (2002),Crime|Drama,0.285714,0.873488,0.937173,0.000000,0.962398,0.188679,0.000000,1.25,...,0.075967,0.013970,0.000000,0.000000,0.00000,semantic,"psychology, surrealism, depressing, dark","dark, psychology, surrealism, depressing, powe...",,0.585880
1,As Good as It Gets (1997),Comedy|Drama|Romance,0.421053,0.851711,0.821408,0.000000,0.870067,0.301887,0.000000,1.25,...,0.075967,0.022353,0.000000,0.000000,0.00000,semantic,"psychology, psychological, feel-good, mental i...","psychology, psychological, feel-good, mental i...",,0.543665
2,"Elephant Man, The (1980)",Drama,0.210526,0.916695,0.950553,0.000000,0.974684,0.150943,0.257252,1.25,...,0.074102,0.011176,0.065604,0.000000,0.00349,semantic,"surrealism, morality, depressing, tragedy, atm...","surrealism, atmospheric, morality, depressing,...",,0.528010
3,What's Eating Gilbert Grape (1993),Drama,0.210526,0.900795,0.845259,0.000000,0.889427,0.150943,0.069812,1.25,...,0.075967,0.011176,0.017803,0.000000,0.00000,semantic,"creepy, depressing, bittersweet, whimsical, me...","creepy, depressing, bittersweet, whimsical, me...",,0.526354
4,3-Iron (Bin-jip) (2004),Drama|Romance,0.263158,0.778431,0.839151,0.000000,0.883842,0.207547,0.000000,1.25,...,0.075967,0.015367,0.000000,0.000000,0.00000,semantic,"existentialism, magical realism, cerebral","existentialism, stylized, magical realism, cer...",,0.510630
5,Underground (1995),Comedy|Drama|War,0.383459,0.880747,0.857766,0.000000,0.905063,0.245283,0.093880,1.25,...,0.075967,0.018161,0.023941,0.000000,0.00000,semantic,"dreamlike, surrealism, magical realism, cerebral","dreamlike, surrealism, magical realism, cerebr...",,0.508756
6,About a Boy (2002),Comedy|Drama|Romance,0.421053,0.921880,0.840314,0.060681,0.881236,0.301887,0.069812,1.25,...,0.075967,0.022353,0.017803,0.000000,0.00000,semantic,"heartwarming, narrated, single mother, whimsic...","heartwarming, narrated, single mother, whimsic...",mother-son relationship,0.507326
7,Song of the Sea (2014),Animation|Children|Fantasy,0.150376,0.552368,0.687609,0.110007,0.733433,0.075472,0.008129,1.25,...,0.075967,0.005588,0.002073,0.000000,0.00000,semantic,"dreamlike, atmosphere","dreamlike, sad, atmosphere, style over substance",grief,0.504611
8,Chicken Run (2000),Animation|Children|Comedy,0.248120,0.590391,0.715241,0.000000,0.774758,0.094340,0.028212,1.25,...,0.075967,0.006985,0.007194,0.000000,0.00000,semantic,"nostalgic, childhood","nostalgic, witty, childhood, life & death",,0.504485
9,"Imposter, The (2012)",Documentary,0.000000,0.725544,0.757417,0.000000,0.806031,0.000000,0.000000,1.25,...,0.075967,0.000000,0.000000,0.000000,0.00000,semantic,"tense, psychological","tense, psychological, twist",,0.499162


dominant_signal
semantic    50
Name: count, dtype: int64

## 15. Diversidad

In [18]:
def main_genre_from_genres(genres):
    genre_list = split_genres(genres)
    return genre_list[0] if genre_list else "Unknown"


def _row_text_terms(row):
    pieces = [
        row.get("core_semantic_explanation_terms", ""),
        row.get("semantic_explanation_terms", ""),
    ]
    return " ".join(str(piece).lower() for piece in pieces if pd.notna(piece))


def assign_semantic_branch(row):
    genres = set(split_genres(row.get("genres", "")))
    main_genre = row.get("main_genre", "")
    if pd.notna(main_genre) and main_genre:
        genres.add(str(main_genre))
    terms = _row_text_terms(row)
    year = pd.to_numeric(row.get("year"), errors="coerce")

    def has_any(keywords):
        return any(keyword in terms for keyword in keywords)

    if "Animation" in genres or "Children" in genres:
        return "animation_family"
    if "Documentary" in genres:
        return "documentary_reflective"
    if has_any(["psychology", "psychological", "mindfuck", "mental illness", "disturbing"]):
        return "psychological_thriller"
    if "Crime" in genres or "Thriller" in genres:
        return "crime_thriller"
    if has_any(["surrealism", "dreamlike", "magical realism", "weird"]):
        return "surreal_fantasy"
    if has_any(["time travel", "artificial intelligence", "space", "dystopia"]):
        return "sci_fi_reflective"
    if has_any(["satire", "satirical", "dark comedy", "black comedy", "witty"]):
        return "comedy_satire"
    if ("Romance" in genres) or ({"Comedy", "Drama"}.issubset(genres)):
        return "romantic_comedy_drama"
    if has_any(["dark", "bleak", "depressing", "tragedy"]):
        return "dark_drama"
    if pd.notna(year) and year < 1980 and "Adventure" in genres:
        return "classic_adventure"
    return "general"


def select_diverse_recommendations(df, top_n=20, max_per_main_genre=3, max_drama_total=5):
    work = df.sort_values("hybrid_score", ascending=False).copy()
    work["main_genre"] = work["genres"].apply(main_genre_from_genres)
    selected_indices = []
    fallback_indices = []
    main_genre_counts = {}
    drama_total = 0

    for idx, row in work.iterrows():
        main_genre = row["main_genre"]
        contains_drama = "Drama" in split_genres(row["genres"])
        if main_genre_counts.get(main_genre, 0) >= max_per_main_genre:
            continue
        if contains_drama and drama_total >= max_drama_total:
            continue
        selected_indices.append(idx)
        main_genre_counts[main_genre] = main_genre_counts.get(main_genre, 0) + 1
        if contains_drama:
            drama_total += 1
        if len(selected_indices) >= top_n:
            break

    if len(selected_indices) < top_n:
        warnings.warn("No se alcanzó el top_n solo con restricciones de diversidad; se rellena con fallback por ranking.")
        for idx in work.index:
            if idx not in selected_indices:
                fallback_indices.append(idx)
                selected_indices.append(idx)
            if len(selected_indices) >= top_n:
                break

    selected = work.loc[selected_indices].copy()
    selected["diversity_selection"] = "principal"
    selected.loc[fallback_indices, "diversity_selection"] = "fallback"
    return selected.reset_index(drop=True)


def diversified_rerank(
    candidates,
    top_n=20,
    score_col="hybrid_score",
    genre_col="main_genre",
    branch_col="semantic_branch",
    signal_col="dominant_signal",
    max_per_genre=4,
    max_per_branch=4,
    max_per_signal=14,
    candidate_pool_size=300,
):
    work = candidates.sort_values(score_col, ascending=False).head(candidate_pool_size).copy()
    if genre_col not in work.columns:
        work[genre_col] = work["genres"].apply(main_genre_from_genres)
    if branch_col not in work.columns:
        work[branch_col] = work.apply(assign_semantic_branch, axis=1)
    if signal_col not in work.columns:
        work[signal_col] = "unknown"

    selected_indices = []
    reasons = {}
    selected_set = set()

    def pass_select(genre_limit, branch_limit, signal_limit, reason):
        genre_counts = work.loc[selected_indices, genre_col].value_counts().to_dict() if selected_indices else {}
        branch_counts = work.loc[selected_indices, branch_col].value_counts().to_dict() if selected_indices else {}
        signal_counts = work.loc[selected_indices, signal_col].value_counts().to_dict() if selected_indices else {}
        for idx, row in work.iterrows():
            if idx in selected_set:
                continue
            genre = row.get(genre_col, "Unknown")
            branch = row.get(branch_col, "general")
            signal = row.get(signal_col, "unknown")
            if genre_counts.get(genre, 0) >= genre_limit:
                continue
            if branch_counts.get(branch, 0) >= branch_limit:
                continue
            if signal_counts.get(signal, 0) >= signal_limit:
                continue
            selected_indices.append(idx)
            selected_set.add(idx)
            reasons[idx] = reason
            genre_counts[genre] = genre_counts.get(genre, 0) + 1
            branch_counts[branch] = branch_counts.get(branch, 0) + 1
            signal_counts[signal] = signal_counts.get(signal, 0) + 1
            if len(selected_indices) >= top_n:
                break

    pass_select(max_per_genre, max_per_branch, max_per_signal, "rerank_constraints")
    if len(selected_indices) < top_n:
        pass_select(max_per_genre + 1, max_per_branch + 1, max_per_signal + 3, "rerank_relaxed")
    if len(selected_indices) < top_n:
        for idx in work.index:
            if idx not in selected_set:
                selected_indices.append(idx)
                selected_set.add(idx)
                reasons[idx] = "score_fallback"
            if len(selected_indices) >= top_n:
                break

    selected = work.loc[selected_indices].copy()
    selected["final_rank"] = np.arange(1, len(selected) + 1)
    selected["selected_by_rerank"] = True
    selected["rerank_reason"] = selected.index.map(reasons).fillna("rerank_constraints")
    return selected.sort_values("final_rank").reset_index(drop=True)


def compare_weight_configs(base_df, weight_configs, top_n=20):
    rows = []
    for config_name, config_weights in weight_configs.items():
        scored = compute_hybrid_score(base_df, config_weights)
        scored = scored.sort_values("hybrid_score", ascending=False).reset_index(drop=True)
        selected = select_diverse_recommendations(scored, top_n=top_n, max_per_main_genre=3, max_drama_total=5)
        rows.append(
            {
                "config": config_name,
                "avg_genre_profile_score": selected["genre_profile_score"].mean(),
                "avg_semantic_profile_score": selected["semantic_profile_score"].mean(),
                "avg_negative_genre_score": selected["negative_genre_score"].mean(),
                "avg_negative_semantic_score": selected["negative_semantic_score"].mean(),
                "avg_item_item_collab_score": selected["item_item_collab_score"].mean(),
                "avg_item_item_negative_collab_score": selected["item_item_negative_collab_score"].mean(),
                "avg_rating_mean": selected["rating_mean"].mean(),
                "avg_rating_count": selected["rating_count"].mean(),
                "avg_popularity_score": selected["popularity_score"].mean(),
                "avg_year_affinity_score": selected["year_affinity_score"].mean() if "year_affinity_score" in selected.columns else np.nan,
                "avg_temporal_mismatch_penalty": selected["temporal_mismatch_penalty"].mean() if "temporal_mismatch_penalty" in selected.columns else np.nan,
                "n_temporal_outliers": selected["is_temporal_outlier"].sum() if "is_temporal_outlier" in selected.columns else 0,
                "n_distinct_main_genres": selected["main_genre"].nunique(),
                "dominant_signal_counts": selected["dominant_signal"].value_counts().to_dict(),
                "top_titles": "; ".join(selected["title"].head(5).astype(str)),
            }
        )
    return pd.DataFrame(rows)


if "main_genre" not in candidates_scored.columns:
    candidates_scored["main_genre"] = candidates_scored["genres"].apply(main_genre_from_genres)
candidates_scored["semantic_branch"] = candidates_scored.apply(assign_semantic_branch, axis=1)

print("Distribución de semantic_branch en candidatos:")
display(candidates_scored["semantic_branch"].value_counts(dropna=False))

pre_rerank_top = candidates_scored.sort_values("hybrid_score", ascending=False).head(20).copy()
pre_rerank_titles = set(pre_rerank_top["title"].astype(str))

weight_configs_for_comparison = dict(WEIGHT_CONFIGS)
weight_configs_for_comparison["adaptive"] = adaptive_weights
display(compare_weight_configs(candidates_scored_base, weight_configs_for_comparison, top_n=20))

recommendations = diversified_rerank(
    candidates_scored,
    top_n=20,
    score_col="hybrid_score",
    genre_col="main_genre",
    branch_col="semantic_branch",
    signal_col="dominant_signal",
    max_per_genre=4,
    max_per_branch=4,
    max_per_signal=14,
    candidate_pool_size=300,
)
recommendations = recommendations.sort_values("final_rank").reset_index(drop=True)

post_rerank_titles = set(recommendations["title"].astype(str))
rerank_entered_titles = sorted(post_rerank_titles - pre_rerank_titles)
rerank_removed_titles = sorted(pre_rerank_titles - post_rerank_titles)
rerank_diagnostic_rows = [
    {"metric": "avg_hybrid_score_pre_rerank_top20", "value": pre_rerank_top["hybrid_score"].mean()},
    {"metric": "avg_hybrid_score_post_rerank_top20", "value": recommendations["hybrid_score"].mean()},
    {"metric": "titles_entered_by_rerank", "value": "; ".join(rerank_entered_titles)},
    {"metric": "titles_removed_by_rerank", "value": "; ".join(rerank_removed_titles)},
    {"metric": "main_genre_distribution_top20", "value": recommendations["main_genre"].value_counts().to_dict()},
    {"metric": "semantic_branch_distribution_top20", "value": recommendations["semantic_branch"].value_counts().to_dict()},
    {"metric": "dominant_signal_distribution_top20", "value": recommendations["dominant_signal"].value_counts().to_dict()},
    {"metric": "decade_distribution_top20", "value": ((pd.to_numeric(recommendations["year"], errors="coerce") // 10) * 10).astype("Int64").value_counts().sort_index().to_dict()},
]
rerank_diagnostics_df = pd.DataFrame(rerank_diagnostic_rows)
display(rerank_diagnostics_df)
rerank_diagnostics_df.to_csv(REPORTS_RESULTADOS / "diagnostico_reranking_diversificado.csv", index=False)

print("Distribución de main_genre en top 20:")
display(recommendations["main_genre"].value_counts(dropna=False))
print("Distribución de semantic_branch en top 20:")
display(recommendations["semantic_branch"].value_counts(dropna=False))
print("Distribución de dominant_signal en top 20:")
display(recommendations["dominant_signal"].value_counts(dropna=False))
print("Distribución por década en top 20:")
display(((pd.to_numeric(recommendations["year"], errors="coerce") // 10) * 10).astype("Int64").value_counts().sort_index())

display(recommendations[["final_rank", "title", "main_genre", "semantic_branch", "dominant_signal", "hybrid_score", "rerank_reason"]])


Distribución de semantic_branch en candidatos:


semantic_branch
general                   1083
crime_thriller            1060
romantic_comedy_drama      725
psychological_thriller     440
animation_family           362
documentary_reflective     242
surreal_fantasy            222
comedy_satire              191
dark_drama                 120
sci_fi_reflective           52
classic_adventure           29
Name: count, dtype: int64

,config,avg_genre_profile_score,avg_semantic_profile_score,avg_negative_genre_score,avg_negative_semantic_score,avg_item_item_collab_score,avg_item_item_negative_collab_score,avg_rating_mean,avg_rating_count,avg_popularity_score,avg_year_affinity_score,avg_temporal_mismatch_penalty,n_temporal_outliers,n_distinct_main_genres,dominant_signal_counts,top_titles
0,personalizado_semantico,0.202256,0.731352,0.153774,0.025701,0.708960,0.000000,3.841404,8676.55,0.474987,0.641281,0.343707,6,10,{'semantic': 20},"25th Hour (2002); Manchurian Candidate, The (1..."
1,equilibrado_semantico,0.211654,0.723298,0.167925,0.042086,0.767739,0.000000,3.853490,9229.70,0.485870,0.627757,0.356664,6,9,{'semantic': 20},"Manchurian Candidate, The (1962); 25th Hour (2..."
2,descubrimiento_semantico,0.200376,0.729303,0.150000,0.025558,0.631490,0.000000,3.825269,7669.45,0.444847,0.610075,0.373607,6,9,{'semantic': 20},"25th Hour (2002); Manchurian Candidate, The (1..."
3,adaptive,0.197744,0.706896,0.159434,0.090373,0.751330,0.025461,3.821023,9914.30,0.485035,0.964708,0.033815,0,9,"{'semantic': 19, 'collab': 1}",25th Hour (2002); As Good as It Gets (1997); E...


,metric,value
0,avg_hybrid_score_pre_rerank_top20,0.503623
1,avg_hybrid_score_post_rerank_top20,0.476359
2,titles_entered_by_rerank,"Emperor's New Groove, The (2000); Lincoln Lawy..."
3,titles_removed_by_rerank,Crimson Tide (1995); Glengarry Glen Ross (1992...
4,main_genre_distribution_top20,"{'Comedy': 4, 'Drama': 4, 'Documentary': 4, 'C..."
5,semantic_branch_distribution_top20,"{'psychological_thriller': 4, 'animation_famil..."
6,dominant_signal_distribution_top20,"{'semantic': 14, 'collab': 6}"
7,decade_distribution_top20,"{1980: 3, 1990: 4, 2000: 9, 2010: 4}"


Distribución de main_genre en top 20:


...

Distribución de dominant_signal en top 20:


dominant_signal
semantic    14
collab       6
Name: count, dtype: int64

Distribución por década en top 20:


year
1980    3
1990    4
2000    9
2010    4
Name: count, dtype: Int64

,final_rank,title,main_genre,semantic_branch,dominant_signal,hybrid_score,rerank_reason
0,1,25th Hour (2002),Crime,psychological_thriller,semantic,0.585880,rerank_constraints
1,2,As Good as It Gets (1997),Comedy,psychological_thriller,semantic,0.543665,rerank_constraints
2,3,"Elephant Man, The (1980)",Drama,surreal_fantasy,semantic,0.528010,rerank_constraints
3,4,What's Eating Gilbert Grape (1993),Drama,psychological_thriller,semantic,0.526354,rerank_constraints
4,5,3-Iron (Bin-jip) (2004),Drama,surreal_fantasy,semantic,0.510630,rerank_constraints
5,6,Underground (1995),Comedy,surreal_fantasy,semantic,0.508756,rerank_constraints
6,7,About a Boy (2002),Comedy,comedy_satire,semantic,0.507326,rerank_constraints
7,8,Song of the Sea (2014),Animation,animation_family,semantic,0.504611,rerank_constraints
8,9,Chicken Run (2000),Animation,animation_family,semantic,0.504485,rerank_constraints
9,10,"Imposter, The (2012)",Documentary,documentary_reflective,semantic,0.499162,rerank_constraints


## 16. Explicaciones

In [19]:
def _clean_explanation_terms(value):
    if value is None:
        return ""
    if isinstance(value, (list, tuple, set)):
        value = ", ".join(str(item) for item in value if item)
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    return str(value).strip()


def build_explanation(row):
    dominant_signal = row.get("dominant_signal", "")
    core_semantic_terms = _clean_explanation_terms(row.get("core_semantic_explanation_terms", ""))
    semantic_terms = core_semantic_terms or _clean_explanation_terms(row.get("semantic_explanation_terms", ""))
    core_negative_terms = _clean_explanation_terms(row.get("core_negative_semantic_terms", ""))
    negative_semantic_terms = core_negative_terms or _clean_explanation_terms(row.get("negative_semantic_terms", ""))
    similar_liked = _clean_explanation_terms(row.get("similar_liked_movies", ""))
    semantic_branch = _clean_explanation_terms(row.get("semantic_branch", ""))
    branch_explanation = {
        "psychological_thriller": "encaja con un perfil de películas psicológicas o de tensión",
        "animation_family": "encaja con una rama de animación o fantasía presente en tu perfil",
        "documentary_reflective": "encaja con una rama más documental o reflexiva",
        "crime_thriller": "encaja con películas de crimen, tensión o investigación",
        "surreal_fantasy": "encaja con rasgos surrealistas, fantásticos o poco convencionales",
    }

    if dominant_signal == "semantic" and semantic_terms:
        explanation = f"Recomendada porque comparte rasgos semánticos especialmente representativos de tus películas mejor valoradas: {semantic_terms}."
    elif dominant_signal == "genre":
        explanation = "Recomendada porque encaja con los géneros que mejor aparecen en tu perfil."
    elif dominant_signal == "collab" and similar_liked:
        explanation = "Recomendada porque presenta similitud colaborativa con películas que valoraste positivamente: " + similar_liked + "."
    elif dominant_signal == "rating":
        explanation = "Recomendada porque destaca por su valoración media en MovieLens."
    elif dominant_signal == "popularity":
        explanation = "Recomendada porque combina afinidad con una señal alta de popularidad en MovieLens."
    elif semantic_terms:
        explanation = f"Recomendada porque comparte algunos rasgos semánticos con tus películas mejor valoradas: {semantic_terms}."
    else:
        explanation = "Recomendada porque mantiene un equilibrio razonable entre afinidad, calidad y popularidad."

    if semantic_branch in branch_explanation:
        explanation += " También " + branch_explanation[semantic_branch] + "."

    negative_semantic_signal = max(row.get("semantic_core_negative_score", 0), row.get("negative_semantic_score", 0))
    if row.get("rating_score", 0) >= 0.65:
        explanation += " Además, tiene buena valoración media."
    if negative_semantic_signal > 0.35 and negative_semantic_terms:
        explanation += f" Se controla la similitud con rasgos presentes en películas peor valoradas: {negative_semantic_terms}."
    if row.get("item_item_negative_collab_score", 0) > 0.35:
        explanation += " También se aplica una penalización colaborativa por similitud con películas peor valoradas."
    return explanation


recommendations["explanation"] = recommendations.apply(build_explanation, axis=1)

## 17. Resultados

In [20]:
RESULT_COLUMNS = [
    "title",
    "year",
    "genres",
    "rating_mean",
    "rating_count",
    "genre_profile_score",
    "semantic_raw_score",
    "negative_semantic_raw_score",
    "semantic_profile_score",
    "negative_genre_score",
    "negative_semantic_score",
    "semantic_explanation_terms",
    "negative_semantic_terms",
    "content_profile_score",
    "negative_similarity_score",
    "item_item_collab_score",
    "item_item_negative_collab_score",
    "rating_score",
    "popularity_score",
    "contrib_genre",
    "contrib_semantic",
    "contrib_collab",
    "contrib_rating",
    "contrib_popularity",
    "penalty_negative_genre",
    "penalty_negative_semantic",
    "penalty_negative_collab",
    "dominant_signal",
    "hybrid_score",
    "explanation",
]

display(recommendations[RESULT_COLUMNS].head(20))

,title,year,genres,rating_mean,rating_count,genre_profile_score,semantic_raw_score,negative_semantic_raw_score,semantic_profile_score,negative_genre_score,...,contrib_semantic,contrib_collab,contrib_rating,contrib_popularity,penalty_negative_genre,penalty_negative_semantic,penalty_negative_collab,dominant_signal,hybrid_score,explanation
0,25th Hour (2002),2002.0,Crime|Drama,3.804011,6332,0.285714,163.462786,0.000000,0.873488,0.188679,...,0.288719,0.162756,0.048241,0.006659,0.013970,0.000000,0.000000,semantic,0.585880,Recomendada porque comparte rasgos semánticos ...
1,As Good as It Gets (1997),1997.0,Comedy|Drama|Romance,3.771600,27176,0.421053,151.158264,0.000000,0.851711,0.301887,...,0.261020,0.147090,0.046296,0.009844,0.022353,0.000000,0.000000,semantic,0.543665,Recomendada porque comparte rasgos semánticos ...
2,"Elephant Man, The (1980)",1980.0,Drama,3.912803,7959,0.210526,192.425317,3.202929,0.916695,0.150943,...,0.292405,0.166945,0.054768,0.007159,0.011176,0.065604,0.000000,semantic,0.528010,Recomendada porque comparte rasgos semánticos ...
3,What's Eating Gilbert Grape (1993),1993.0,Drama,3.747386,19510,0.210526,179.890987,1.515045,0.900795,0.150943,...,0.266828,0.145675,0.044843,0.009119,0.011176,0.017803,0.000000,semantic,0.526354,Recomendada porque comparte rasgos semánticos ...
4,3-Iron (Bin-jip) (2004),2004.0,Drama|Romance,3.949209,1644,0.263158,119.115993,0.000000,0.778431,0.207547,...,0.265153,0.108087,0.056953,0.003713,0.015367,0.000000,0.000000,semantic,0.510630,Recomendada porque comparte rasgos semánticos ...
5,Underground (1995),1995.0,Comedy|Drama|War,3.962975,1499,0.383459,166.338725,1.665823,0.880747,0.245283,...,0.271519,0.118586,0.057779,0.003511,0.018161,0.023941,0.000000,semantic,0.508756,Recomendada porque comparte rasgos semánticos ...
6,About a Boy (2002),2002.0,Comedy|Drama|Romance,3.626879,12969,0.421053,196.077470,1.515045,0.921880,0.301887,...,0.264371,0.135503,0.037613,0.008227,0.022353,0.017803,0.000000,semantic,0.507326,Recomendada porque comparte rasgos semánticos ...
7,Song of the Sea (2014),2014.0,Animation|Children|Fantasy,4.019345,1680,0.150376,70.421241,1.276329,0.552368,0.075472,...,0.220030,0.142140,0.061161,0.003760,0.005588,0.002073,0.000000,semantic,0.504611,Recomendada porque comparte rasgos semánticos ...
8,Chicken Run (2000),2000.0,Animation|Children|Comedy,3.468736,22822,0.248120,78.275309,1.290081,0.590391,0.094340,...,0.232427,0.157480,0.028124,0.009462,0.006985,0.007194,0.000000,semantic,0.504485,Recomendada porque comparte rasgos semánticos ...
9,"Imposter, The (2012)",2012.0,Documentary,3.754595,925,0.000000,105.172847,0.000000,0.725544,0.000000,...,0.241809,0.133654,0.045276,0.002457,0.000000,0.000000,0.000000,semantic,0.499162,Recomendada porque comparte rasgos semánticos ...


## 18. Sanity checks finales

In [21]:
n_recommendations = len(recommendations)
already_watched = recommendations["movieId"].isin(watched_movie_ids).sum()
already_rated = recommendations["movieId"].isin(rated_movie_ids).sum()
mean_rating_top = recommendations["rating_mean"].mean()
mean_count_top = recommendations["rating_count"].mean()
distinct_main_genres = recommendations["main_genre"].nunique()
high_negative = (recommendations["negative_similarity_score"] > 0.35).sum()
high_negative_collab = (recommendations["item_item_negative_collab_score"] > 0.35).sum()
high_semantic = (recommendations["semantic_profile_score"] > 0.5).sum()
high_negative_semantic = (recommendations["negative_semantic_score"] > 0.35).sum()
high_core_semantic = (recommendations["semantic_core_score"] > 0.5).sum() if "semantic_core_score" in recommendations.columns else 0
high_core_negative_semantic = (recommendations["semantic_core_negative_score"] > 0.35).sum() if "semantic_core_negative_score" in recommendations.columns else 0
low_rating = (recommendations["rating_mean"] < 3.2).sum()

def print_top_mean(column):
    if column in recommendations.columns:
        print(f"Media {column} top 20: {recommendations[column].mean():.3f}")


print(f"Número de recomendaciones: {n_recommendations}")
print(f"Ya vistas: {already_watched}")
print(f"Ya valoradas: {already_rated}")
print(f"rating_mean medio top 20: {mean_rating_top:.3f}")
print(f"rating_count medio top 20: {mean_count_top:.1f}")
print(f"Géneros principales distintos: {distinct_main_genres}")
print(f"Recomendaciones con negative_similarity_score > 0.35: {high_negative}")
print(f"Recomendaciones con item_item_negative_collab_score > 0.35: {high_negative_collab}")
print(f"Recomendaciones con semantic_profile_score > 0.5: {high_semantic}")
print(f"Recomendaciones con negative_semantic_score > 0.35: {high_negative_semantic}")
print(f"Recomendaciones con semantic_core_score > 0.5: {high_core_semantic}")
print(f"Recomendaciones con semantic_core_negative_score > 0.35: {high_core_negative_semantic}")
print(f"Recomendaciones con rating_mean < 3.2: {low_rating}")
print("Distribución de dominant_signal en top 20:")
print(recommendations["dominant_signal"].value_counts())
for diagnostic_column in [
    "semantic_profile_score",
    "semantic_core_score",
    "semantic_core_negative_score",
    "semantic_net_score",
    "item_item_collab_score",
    "rating_score",
    "contrib_semantic",
    "contrib_collab",
    "contrib_rating",
    "year_affinity_score",
    "temporal_mismatch_penalty",
]:
    print_top_mean(diagnostic_column)
print("Top términos semánticos positivos del usuario:")
print(top_positive_semantic_terms[:10])
print("Top términos semánticos negativos del usuario:")
print(top_negative_semantic_terms[:10])

if already_watched > 0:
    warnings.warn("Hay recomendaciones que ya estaban vistas en Trakt.")
if already_rated > 0:
    warnings.warn("Hay recomendaciones que ya estaban valoradas en Trakt.")
if low_rating > 0:
    warnings.warn("Hay recomendaciones por debajo del umbral mínimo de rating_mean.")

Número de recomendaciones: 20
Ya vistas: 0
Ya valoradas: 0
rating_mean medio top 20: 3.824
rating_count medio top 20: 8215.0
Géneros principales distintos: 6
Recomendaciones con negative_similarity_score > 0.35: 1
Recomendaciones con item_item_negative_collab_score > 0.35: 2
Recomendaciones con semantic_profile_score > 0.5: 14
Recomendaciones con negative_semantic_score > 0.35: 0
Recomendaciones con semantic_core_score > 0.5: 14
Recomendaciones con semantic_core_negative_score > 0.35: 0
Recomendaciones con rating_mean < 3.2: 0
Distribución de dominant_signal en top 20:
dominant_signal
semantic    14
collab       6
Name: count, dtype: int64
Media semantic_profile_score top 20: 0.592
Media semantic_core_score top 20: 0.670
Media semantic_core_negative_score top 20: 0.019
Media semantic_net_score top 20: 0.710
Media item_item_collab_score top 20: 0.806
Media rating_score top 20: 0.412
Media contrib_semantic top 20: 0.213
Media contrib_collab top 20: 0.145
Media contrib_rating top 20: 0.04

## 19. Exportación

In [22]:
EXPORT_COLUMNS = [
    "movieId",
    "title",
    "year",
    "genres",
    "rating_mean",
    "rating_count",
    "rerank_reason",
    "selected_by_rerank",
    "semantic_branch",
    "final_rank",
    "genre_profile_score",
    "semantic_raw_score",
    "negative_semantic_raw_score",
    "semantic_core_raw_score",
    "semantic_core_negative_raw_score",
    "semantic_core_score",
    "semantic_core_negative_score",
    "semantic_net_raw_score",
    "semantic_net_score",
    "semantic_negative_beta",
    "semantic_profile_score",
    "negative_genre_score",
    "negative_semantic_score",
    "semantic_explanation_terms",
    "core_semantic_explanation_terms",
    "core_negative_semantic_terms",
    "negative_semantic_terms",
    "content_profile_score",
    "negative_similarity_score",
    "item_item_collab_score",
    "item_item_negative_collab_score",
    "positive_collab_score",
    "negative_collab_score",
    "rating_score",
    "popularity_score",
    "allow_temporal_outlier",
    "is_temporal_outlier",
    "temporal_distance_from_profile",
    "temporal_mismatch_penalty",
    "year_affinity_score",
    "contrib_genre",
    "contrib_semantic",
    "contrib_collab",
    "contrib_rating",
    "contrib_popularity",
    "contrib_year_affinity",
    "penalty_negative_genre",
    "penalty_negative_semantic",
    "penalty_negative_collab",
    "penalty_temporal_mismatch",
    "dominant_signal",
    "hybrid_score",
    "n_collab_evidence",
    "similar_liked_movies",
    "similar_disliked_movies",
    "explanation",
]

export_path = REPORTS_RESULTADOS / "recomendaciones_hibridas_item_item.csv"
recommendations[EXPORT_COLUMNS].to_csv(export_path, index=False)
print(f"Recomendaciones exportadas en: {export_path}")

Recomendaciones exportadas en: ..\reports\resultados\recomendaciones_hibridas_item_item.csv


In [23]:
diagnostic_cols = [
    "title",
    "year",
    "genres",
    "rating_mean",
    "rating_count",
    "rerank_reason",
    "selected_by_rerank",
    "semantic_branch",
    "final_rank",
    "genre_profile_score",
    "semantic_profile_score",
    "semantic_core_score",
    "semantic_core_negative_score",
    "semantic_net_score",
    "negative_genre_score",
    "negative_semantic_score",
    "item_item_collab_score",
    "item_item_negative_collab_score",
    "rating_score",
    "popularity_score",
    "allow_temporal_outlier",
    "is_temporal_outlier",
    "temporal_distance_from_profile",
    "temporal_mismatch_penalty",
    "year_affinity_score",
    "contrib_genre",
    "contrib_semantic",
    "contrib_collab",
    "contrib_rating",
    "contrib_popularity",
    "contrib_year_affinity",
    "penalty_negative_genre",
    "penalty_negative_semantic",
    "penalty_negative_collab",
    "penalty_temporal_mismatch",
    "dominant_signal",
    "semantic_explanation_terms",
    "core_semantic_explanation_terms",
    "core_negative_semantic_terms",
    "negative_semantic_terms",
    "hybrid_score",
    "explanation",
]

available_cols = [c for c in diagnostic_cols if c in recommendations.columns]

recommendations[available_cols].to_csv(
    REPORTS_RESULTADOS / "diagnostico_recomendaciones_top20.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Exportado:", REPORTS_RESULTADOS / "diagnostico_recomendaciones_top20.csv")

Exportado: ..\reports\resultados\diagnostico_recomendaciones_top20.csv


## Conclusión

Este notebook implementa el primer prototipo del recomendador híbrido final.

MovieLens se usa para aprender similitud colaborativa entre películas. Trakt se usa para personalizar el ranking con gustos reales y excluir películas vistas o valoradas. LightFM queda como experimento independiente, no como ranking final.

Próximas mejoras:

1. Reutilizar el perfil avanzado de 04d con géneros + tags.
2. Ajustar pesos del hybrid_score.
3. Evaluar con split train/test por usuario.
4. Mejorar diversidad.
5. Refactorizar funciones a src solo cuando el notebook sea estable.